In [356]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import pickle

import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from datetime import datetime
from pandas.tseries.offsets import MonthEnd

input_table = 'TRN_DF_ECOM_OFFTAKE_CHAIN_PSKU'
month_run = '2026-06-30'
os.listdir('/data/aman_singh/acuuracy_check')

['Heuristics_all_combination_qcom_cp_apr_live.xlsx',
 'chek_nan.csv',
 'Heuristics_all_combination_ecom_mar_live.xlsx',
 'combine_model+missing_forecasts_brand_asm.ipynb',
 'April-26 Plans.xlsx',
 'Heuristics_all_combination_ecom_may_live.xlsx',
 'Heuristics_all_combination_qcom_chain_psku_june_live.xlsx',
 'Norms 202602.csv',
 'Norms 202606.csv',
 'key_check.csv',
 'Stat Demand Forecast MT_as_on_11th_May_2026.xlsb',
 'acc_framework_may_final.xlsx',
 'acc_offtakes_till_may.csv',
 'swigy_vol_chk.csv',
 'ALL Channels Accuracy_fva.ipynb',
 'Marico Ltd._forecast_Jul 2026_to_Oct 2026.csv',
 'all_combination_ecom_backtest_pred.csv',
 'ecom_chain_psku_offtake_to_secondary_v6_PROD.ipynb',
 "gt_channels Live Run may'26.csv",
 'seasonality.xlsx',
 'missing_df_gt_all.csv',
 'ECOM Chain PSKU Primary_as_on_12_Jan_2026 (1).xlsb',
 'SOH - 01 Jun.xlsx',
 'qcom_chain_depot_psku_zepto_inc.xlsx',
 'stat_fva_till_may.csv',
 'Heuristics_all_combination_qcom_depot_psku_june_live.xlsx',
 'QCOM_NORMS_FINAL_of

In [357]:
base_dir = '/data/aman_singh/acuuracy_check/backtest_files'

In [358]:
def list_all_files_in_directory(root):
    out = []

    for path, subdirs, files in os.walk(root):
        for name in files:
            out.append(os.path.join(path, name))

    return out

In [359]:
list_all_files_in_directory(base_dir)

['/data/aman_singh/acuuracy_check/backtest_files/prophet_data_train_till_31_Dec_2025 (10).csv',
 '/data/aman_singh/acuuracy_check/backtest_files/trend_file_train_till_31_Mar_2026 (11).csv',
 '/data/aman_singh/acuuracy_check/backtest_files/prophet_data_train_till_31_Mar_2026 (15).csv',
 '/data/aman_singh/acuuracy_check/backtest_files/trend_file_train_till_28_Feb_2026 (8).csv',
 '/data/aman_singh/acuuracy_check/backtest_files/prophet_data_train_till_31_Mar_2026 (16).csv',
 '/data/aman_singh/acuuracy_check/backtest_files/trend_file_train_till_28_Feb_2026 (9).csv',
 '/data/aman_singh/acuuracy_check/backtest_files/prophet_data_train_till_28_Feb_2026 (9).csv',
 '/data/aman_singh/acuuracy_check/backtest_files/prophet_data_train_till_28_Feb_2026 (10).csv',
 '/data/aman_singh/acuuracy_check/backtest_files/trend_file_train_till_31_Dec_2025 (10).csv',
 '/data/aman_singh/acuuracy_check/backtest_files/trend_file_train_till_31_Jan_2026 (9).csv',
 '/data/aman_singh/acuuracy_check/backtest_files/proph

In [360]:
def discover_channel(file_path):
    # file_path = file_path.split('/')

    # if 'ECOM' in file_path:
    #     return 'ECOM'
    # elif 'QCOM' in file_path:
    #     return 'QCOM'
    # elif 'MT' in file_path:
    #     return 'MT'
    # else:
    #     return 'Channel not found'

    return 'ECOM'


In [361]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [362]:
data_query = f"""
    select * from {input_table}
    where month_date >= '2023-01-01' and run_month = '{month_run}'
        
"""

offtake_df = pd.read_sql(data_query, dev_conn)
offtake_df.head()

,MONTH_DATE,PLATFORM_NAME,PARENT_MATERIAL_CODE,BRAND_CODE,VOL_IN_RUM,INDEXBPM,IMPUTED,BIG_BILLION_DAYS,BIG_BILLION_DAYS_LAG_1,BIG_BILLION_DAYS_LAG_2,BIG_BILLION_DAYS_LEAD_1,BIG_BILLION_DAYS_LEAD_2,GREAT_INDIAN_FESTIVAL,GREAT_INDIAN_FESTIVAL_LAG_1,GREAT_INDIAN_FESTIVAL_LAG_2,GREAT_INDIAN_FESTIVAL_LEAD_1,GREAT_INDIAN_FESTIVAL_LEAD_2,RUN_MONTH
0,2023-01-31,Amazon ARIPL,718288,SAFF GOLD,9.640,13.27071,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
1,2023-02-28,Amazon ARIPL,718288,SAFF GOLD,7.915,10.89602,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
2,2023-03-31,Amazon ARIPL,718288,SAFF GOLD,9.505,13.08486,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
3,2023-04-30,Amazon ARIPL,718288,SAFF GOLD,9.290,12.78889,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
4,2023-05-31,Amazon ARIPL,718288,SAFF GOLD,8.050,11.08187,0,0,0,0,0,0,0,0,0,0,0,2026-06-30


In [363]:
offtake_df.columns = offtake_df.columns.str.lower()

In [364]:
offtake_df.duplicated(
    subset=['platform_name','parent_material_code', 'month_date']).sum()

0

In [365]:
offtake_df['brand_code'] = np.where(
    ((offtake_df['parent_material_code'] == 715096) &
    (offtake_df['brand_code'] == 'CO_SO_PCP')),
    'CO_SO_FS',
    offtake_df['brand_code']
)

In [366]:
# offtake_df = offtake_df[offtake_df['platform_name'].isin(
#     ['Amazon', 'Big Basket', 'Flipkart Grocery', 'Flipkart National'])]

In [367]:
offtake_df['key'] = offtake_df[['platform_name','parent_material_code']].astype(str).agg('_'.join, axis=1)
# offtake_df.rename(columns={'realigned_psku': 'parent_material_code'}, inplace=True)
# offtake_df.drop([ 'run_month'], axis=1, inplace=True)
offtake_df['parent_material_code'] = offtake_df['parent_material_code'].astype(int)

In [368]:
offtake_df.duplicated(subset=['key', 'month_date']).sum()

0

In [369]:
(offtake_df['key'] == offtake_df[['platform_name','parent_material_code']].astype(str).agg('_'.join, axis=1)).all()

True

In [370]:
# realigned_df.to_csv('OT_data_debug.csv', index=False)

### Collate MIL

In [371]:
base_dir

'/data/aman_singh/acuuracy_check/backtest_files'

In [372]:
def collate_file(file_hint, extension='.csv'):
    collated_file = pd.DataFrame()

    run_path = f'{base_dir}'
    all_files = list_all_files_in_directory(run_path)

    for file_path in all_files:
        if file_hint in file_path:
            if extension == '.csv':
                print(file_path)
                read_file = pd.read_csv(file_path)
                # read_file['channel'] = discover_channel(file_path)
                read_file['run'] = 'run'
                read_file['step'] = file_path.split('/')[3]
                read_file['file_path'] = file_path

                collated_file = pd.concat(
                    [collated_file, read_file]
                )
                del read_file

    return collated_file

In [373]:
trend_file_df = collate_file('trend_file_train_till')
prophet_file_df = collate_file('prophet_data_train_till')

/data/aman_singh/acuuracy_check/backtest_files/trend_file_train_till_31_Mar_2026 (11).csv
/data/aman_singh/acuuracy_check/backtest_files/trend_file_train_till_28_Feb_2026 (8).csv
/data/aman_singh/acuuracy_check/backtest_files/trend_file_train_till_28_Feb_2026 (9).csv
/data/aman_singh/acuuracy_check/backtest_files/trend_file_train_till_31_Dec_2025 (10).csv
/data/aman_singh/acuuracy_check/backtest_files/trend_file_train_till_31_Jan_2026 (9).csv
/data/aman_singh/acuuracy_check/backtest_files/trend_file_train_till_31_Dec_2025 (24).csv
/data/aman_singh/acuuracy_check/backtest_files/trend_file_train_till_31_Mar_2026 (10).csv
/data/aman_singh/acuuracy_check/backtest_files/trend_file_train_till_31_Jan_2026 (22).csv
/data/aman_singh/acuuracy_check/backtest_files/prophet_data_train_till_31_Dec_2025 (10).csv
/data/aman_singh/acuuracy_check/backtest_files/prophet_data_train_till_31_Mar_2026 (15).csv
/data/aman_singh/acuuracy_check/backtest_files/prophet_data_train_till_31_Mar_2026 (16).csv
/data/a

In [374]:
# forecast_train_till_file_df = collate_file('forecast_train_till_')

In [375]:
# forecast_train_till_file_df

In [376]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,vol_in_rum,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,pred_best_model,pred_value_best_model
0,Big Basket_715100,2023-01-31,0.000000,0.5,0.683333,0.322004,0.326167,0.000000,0.000061,0.000083,0.000039,0.000040,715100,Big Basket,0.4,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.000000,1,1220.081000,0.000049,0.4,0.000049,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Big Basket_715100,2023-02-28,0.400000,0.5,0.683333,0.483122,0.386167,0.000049,0.000061,0.000083,0.000059,0.000047,715100,Big Basket,0.4,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.000000,1,1220.081000,0.000049,0.4,0.000049,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Big Basket_715100,2023-03-31,0.400000,0.5,0.683333,0.392078,0.652333,0.000049,0.000061,0.000083,0.000048,0.000080,715100,Big Basket,0.7,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.583333,1,1220.081000,0.000085,0.7,0.000085,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Big Basket_715100,2023-04-30,0.517081,0.5,0.683333,0.520934,0.632333,0.000063,0.000061,0.000083,0.000064,0.000077,715100,Big Basket,0.7,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.875000,2,1220.081000,0.000085,0.7,0.000085,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Big Basket_715100,2023-05-31,0.578341,0.6,0.683333,0.475287,0.774000,0.000071,0.000073,0.000083,0.000058,0.000094,715100,Big Basket,1.2,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,2.400000,2,1220.081000,0.000146,1.2,0.000146,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75627,Nykaa_810605,2026-05-31,0.000000,0.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,810605,Nykaa,0.0,KAYA_ML,0.0,0.0,0.0,0.0,0.0,0.000000,2,1226.374229,0.000000,0.0,0.000000,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75628,Nykaa_810605,2026-06-30,0.000000,0.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,810605,Nykaa,0.0,KAYA_ML,0.0,0.0,0.0,0.0,0.0,0.000000,2,1226.374229,0.000000,0.0,0.000000,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75629,Nykaa_810605,2026-07-31,0.000000,0.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,810605,Nykaa,0.0,KAYA_ML,0.0,0.0,0.0,0.0,0.0,NaN,3,1226.374229,0.000000,0.0,0.000000,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75630,Nykaa_810605,2026-08-31,0.000000,0.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,810605,Nykaa,0.0,KAYA_ML,0.0,0.0,0.0,0.0,0.0,NaN,3,1226.374229,0.000000,0.0,0.000000,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [377]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'platform_name', 'vol_in_rum', 'brand_code',
       'big_billion_days', 'big_billion_days_lag_1', 'big_billion_days_lag_2',
       'big_billion_days_lead_1', 'big_billion_days_lead_2', 'ratio_last_year',
       'quarter', 'qtr_ind_rate', 'vol_in_rum_value', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'great_indian_festival', 'great_indian_festival_lag_1',
       'great_indian_festival_lag_2', 'great_indian_festival_lead_1',
       'great_indian_festival_lead_2', 'pred_best_model',
       'pred_value_best_model'],
      dtype='object')

In [378]:
trend_file_df['platform_name'].unique()

array(['Big Basket', 'Flipkart Grocery', 'Flipkart National', 'Myntra',
       'Nykaa', 'Amazon ARIPL', 'Amazon RK'], dtype=object)

In [379]:
trend_file_df['month_date'] = pd.to_datetime(trend_file_df['month_date'])
prophet_file_df['month_date'] = pd.to_datetime(prophet_file_df['month_date'])

trend_file_df['train_till'] = pd.to_datetime(trend_file_df['train_till'])
prophet_file_df['train_till'] = pd.to_datetime(prophet_file_df['train_till'])

trend_file_df['run_month'] = pd.to_datetime(trend_file_df['train_till'] + MonthEnd(1))
prophet_file_df['run_month'] = pd.to_datetime(prophet_file_df['train_till'] + MonthEnd(1))

In [380]:
trend_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum(), \
prophet_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum()

(0, 0)

In [381]:
mappings = {}

for run_month in trend_file_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-04-30 00:00:00'): {Timestamp('2026-04-30 00:00:00'): 'M',
  Timestamp('2026-05-31 00:00:00'): 'M+1',
  Timestamp('2026-06-30 00:00:00'): 'M+2',
  Timestamp('2026-07-31 00:00:00'): 'M+3',
  Timestamp('2026-08-31 00:00:00'): 'M+4',
  Timestamp('2026-09-30 00:00:00'): 'M+5',
  Timestamp('2026-10-31 00:00:00'): 'M+6',
  Timestamp('2026-11-30 00:00:00'): 'M+7',
  Timestamp('2026-12-31 00:00:00'): 'M+8'},
 Timestamp('2026-03-31 00:00:00'): {Timestamp('2026-03-31 00:00:00'): 'M',
  Timestamp('2026-04-30 00:00:00'): 'M+1',
  Timestamp('2026-05-31 00:00:00'): 'M+2',
  Timestamp('2026-06-30 00:00:00'): 'M+3',
  Timestamp('2026-07-31 00:00:00'): 'M+4',
  Timestamp('2026-08-31 00:00:00'): 'M+5',
  Timestamp('2026-09-30 00:00:00'): 'M+6',
  Timestamp('2026-10-31 00:00:00'): 'M+7',
  Timestamp('2026-11-30 00:00:00'): 'M+8'},
 Timestamp('2026-01-31 00:00:00'): {Timestamp('2026-01-31 00:00:00'): 'M',
  Timestamp('2026-02-28 00:00:00'): 'M+1',
  Timestamp('2026-03-31 00:00:00'): 'M+2',

In [382]:
trend_file_df['M month'] = trend_file_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

In [383]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,vol_in_rum,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,pred_best_model,pred_value_best_model,run_month,M month
0,Big Basket_715100,2023-01-31,0.000000,0.5,0.683333,0.322004,0.326167,0.000000,0.000061,0.000083,0.000039,0.000040,715100,Big Basket,0.4,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.000000,1,1220.081000,0.000049,0.4,0.000049,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,None
1,Big Basket_715100,2023-02-28,0.400000,0.5,0.683333,0.483122,0.386167,0.000049,0.000061,0.000083,0.000059,0.000047,715100,Big Basket,0.4,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.000000,1,1220.081000,0.000049,0.4,0.000049,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,None
2,Big Basket_715100,2023-03-31,0.400000,0.5,0.683333,0.392078,0.652333,0.000049,0.000061,0.000083,0.000048,0.000080,715100,Big Basket,0.7,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.583333,1,1220.081000,0.000085,0.7,0.000085,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,None
3,Big Basket_715100,2023-04-30,0.517081,0.5,0.683333,0.520934,0.632333,0.000063,0.000061,0.000083,0.000064,0.000077,715100,Big Basket,0.7,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.875000,2,1220.081000,0.000085,0.7,0.000085,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,None
4,Big Basket_715100,2023-05-31,0.578341,0.6,0.683333,0.475287,0.774000,0.000071,0.000073,0.000083,0.000058,0.000094,715100,Big Basket,1.2,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,2.400000,2,1220.081000,0.000146,1.2,0.000146,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75627,Nykaa_810605,2026-05-31,0.000000,0.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,810605,Nykaa,0.0,KAYA_ML,0.0,0.0,0.0,0.0,0.0,0.000000,2,1226.374229,0.000000,0.0,0.000000,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28,M+3
75628,Nykaa_810605,2026-06-30,0.000000,0.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,810605,Nykaa,0.0,KAYA_ML,0.0,0.0,0.0,0.0,0.0,0.000000,2,1226.374229,0.000000,0.0,0.000000,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28,M+4
75629,Nykaa_810605,2026-07-31,0.000000,0.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,810605,Nykaa,0.0,KAYA_ML,0.0,0.0,0.0,0.0,0.0,NaN,3,1226.374229,0.000000,0.0,0.000000,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28,M+5
75630,Nykaa_810605,2026-08-31,0.000000,0.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,810605,Nykaa,0.0,KAYA_ML,0.0,0.0,0.0,0.0,0.0,NaN,3,1226.374229,0.000000,0.0,0.000000,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28,M+6


In [384]:
trend_file_df[trend_file_df['M month'].notna()]

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,vol_in_rum,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,pred_best_model,pred_value_best_model,run_month,M month
39,Big Basket_715100,2026-04-30,0.0016,0.0,0.0,0.0,0.006667,1.952022e-07,0.0,0.0,0.0,8.133873e-07,715100,Big Basket,0.0,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.875000,2,1220.081000,0.0,0.0,0.0,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,M
40,Big Basket_715100,2026-05-31,0.0016,0.0,0.0,0.0,0.006667,1.952022e-07,0.0,0.0,0.0,8.133873e-07,715100,Big Basket,0.0,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,2.400000,2,1220.081000,0.0,0.0,0.0,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,M+1
41,Big Basket_715100,2026-06-30,0.0016,0.0,0.0,0.0,0.085333,1.952022e-07,0.0,0.0,0.0,1.041136e-05,715100,Big Basket,0.0,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,1.166667,2,1220.081000,0.0,0.0,0.0,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,M+2
42,Big Basket_715100,2026-07-31,0.0016,0.0,0.0,0.0,0.015000,1.952022e-07,0.0,0.0,0.0,1.830122e-06,715100,Big Basket,0.0,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.000000,3,1220.081000,0.0,0.0,0.0,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,M+3
43,Big Basket_715100,2026-08-31,0.0016,0.0,0.0,0.0,0.000000,1.952022e-07,0.0,0.0,0.0,0.000000e+00,715100,Big Basket,0.0,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.115385,3,1220.081000,0.0,0.0,0.0,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,M+4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75627,Nykaa_810605,2026-05-31,0.0000,0.0,0.0,NaN,0.000000,0.000000e+00,0.0,0.0,NaN,0.000000e+00,810605,Nykaa,0.0,KAYA_ML,0.0,0.0,0.0,0.0,0.0,0.000000,2,1226.374229,0.0,0.0,0.0,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28,M+3
75628,Nykaa_810605,2026-06-30,0.0000,0.0,0.0,NaN,0.000000,0.000000e+00,0.0,0.0,NaN,0.000000e+00,810605,Nykaa,0.0,KAYA_ML,0.0,0.0,0.0,0.0,0.0,0.000000,2,1226.374229,0.0,0.0,0.0,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28,M+4
75629,Nykaa_810605,2026-07-31,0.0000,0.0,0.0,NaN,0.000000,0.000000e+00,0.0,0.0,NaN,0.000000e+00,810605,Nykaa,0.0,KAYA_ML,0.0,0.0,0.0,0.0,0.0,NaN,3,1226.374229,0.0,0.0,0.0,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28,M+5
75630,Nykaa_810605,2026-08-31,0.0000,0.0,0.0,NaN,0.000000,0.000000e+00,0.0,0.0,NaN,0.000000e+00,810605,Nykaa,0.0,KAYA_ML,0.0,0.0,0.0,0.0,0.0,NaN,3,1226.374229,0.0,0.0,0.0,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28,M+6


In [385]:
trend_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-01-31,2025-12-31
0,2026-02-28,2026-01-31
0,2026-03-31,2026-02-28
0,2026-04-30,2026-03-31


In [386]:
prophet_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-01-31,2025-12-31
0,2026-02-28,2026-01-31
0,2026-03-31,2026-02-28
0,2026-04-30,2026-03-31


In [387]:
trend_file_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7'],
      dtype=object)

In [388]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")

In [389]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [390]:
trend_file_df['portfolio'].isna().sum()

0

In [391]:
prophet_file_df[
    ['month_date', 'key', 'run_month']
].duplicated().sum()

0

In [392]:
prophet_file_df

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,yhat_60_%ile,yhat_70_%ile,yhat_75_%ile,trend_60_%ile,trend_70_%ile,trend_75_%ile,additive_terms,additive_terms_lower,additive_terms_upper,extra_regressors_additive,extra_regressors_additive_lower,extra_regressors_additive_upper,great_indian_festival,great_indian_festival_lower,great_indian_festival_upper,great_indian_festival_lag_1,great_indian_festival_lag_1_lower,great_indian_festival_lag_1_upper,great_indian_festival_lag_2,great_indian_festival_lag_2_lower,great_indian_festival_lag_2_upper,great_indian_festival_lead_1,great_indian_festival_lead_1_lower,great_indian_festival_lead_1_upper,great_indian_festival_lead_2,great_indian_festival_lead_2_lower,great_indian_festival_lead_2_upper,ratio_last_year,ratio_last_year_lower,ratio_last_year_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat,key,y,month_date,brand_code,qtr_ind_rate,vol_in_rum,vol_in_rum_value,yhat_value,Model_Run,Model_Type,type,train_till,run,step,file_path,big_billion_days,big_billion_days_lower,big_billion_days_upper,big_billion_days_lag_1,big_billion_days_lag_1_lower,big_billion_days_lag_1_upper,big_billion_days_lag_2,big_billion_days_lag_2_lower,big_billion_days_lag_2_upper,big_billion_days_lead_1,big_billion_days_lead_1_lower,big_billion_days_lead_1_upper,big_billion_days_lead_2,big_billion_days_lead_2_lower,big_billion_days_lead_2_upper,run_month
0,2023-01-31,6.694945,8.337735,11.289555,6.694945,6.694945,10.156462,10.487147,10.660259,6.694945,6.694945,6.694945,3.188692,3.188692,3.188692,0.431220,0.431220,0.431220,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.431220,0.431220,0.431220,2.757472,2.757472,2.757472,0.0,0.0,0.0,9.883636,Amazon ARIPL_718288,9.640,2023-01-31,SAFF GOLD,138865.260689,9.640,0.133866,0.137249,Yes,prophet,training,2025-12-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-31
1,2023-02-28,6.789457,6.492393,9.275569,6.789457,6.789457,8.119195,8.433829,8.605125,6.789457,6.789457,6.789457,1.045889,1.045889,1.045889,1.212099,1.212099,1.212099,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.212099,1.212099,1.212099,-0.166211,-0.166211,-0.166211,0.0,0.0,0.0,7.835346,Amazon ARIPL_718288,7.915,2023-02-28,SAFF GOLD,138865.260689,7.915,0.109912,0.108806,Yes,prophet,training,2025-12-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-31
2,2023-03-31,6.894095,7.532693,10.399542,6.894095,6.894095,9.278415,9.570508,9.723756,6.894095,6.894095,6.894095,2.066447,2.066447,2.066447,-0.640276,-0.640276,-0.640276,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.640276,-0.640276,-0.640276,2.706723,2.706723,2.706723,0.0,0.0,0.0,8.960542,Amazon ARIPL_718288,9.505,2023-03-31,SAFF GOLD,138865.260689,9.505,0.131991,0.124431,Yes,prophet,training,2025-12-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-31
3,2023-04-30,6.995358,6.298184,9.365944,6.995358,6.995358,8.177136,8.478038,8.684751,6.995358,6.995358,6.995358,0.910402,0.910402,0.910402,1.373025,1.373025,1.373025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.373025,1.373025,1.373025,-0.462623,-0.462623,-0.462623,0.0,0.0,0.0,7.905760,Amazon ARIPL_718288,9.290,2023-04-30,SAFF GOLD,138865.260689,9.290,0.129006,0.109784,Yes,prophet,training,2025-12-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-31
4,2023-05-31,7.099996,5.888849,8.744782,7.099996,7.099996,7.603606,7.909299,8.098855,7.099996,7.099996,7.099996,0.226748,0.226748,0.226748,0.650903,0.650903,0.650903,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.650903,0.650903,0.650903,-0.424155,-0.424155,-0.424155,0.0,

In [393]:
# Merge 70th percentile Prophet predictions
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    prophet_file_df[['month_date', 'key', 'run_month', 'yhat_70_%ile','yhat_60_%ile']].rename(
        columns={
            'yhat_70_%ile': 'pred_prophet_70%ile',
            'yhat_60_%ile': 'pred_prophet_60%ile'
        }
    ),
    on=['month_date', 'key', 'run_month'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [394]:
assert trend_file_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0

In [395]:
trend_file_df.drop('vol_in_rum', axis=1, inplace=True)

In [396]:
offtake_df.head()

,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,indexbpm,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,key
0,2023-01-31,Amazon ARIPL,718288,SAFF GOLD,9.640,13.27071,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,Amazon ARIPL_718288
1,2023-02-28,Amazon ARIPL,718288,SAFF GOLD,7.915,10.89602,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,Amazon ARIPL_718288
2,2023-03-31,Amazon ARIPL,718288,SAFF GOLD,9.505,13.08486,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,Amazon ARIPL_718288
3,2023-04-30,Amazon ARIPL,718288,SAFF GOLD,9.290,12.78889,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,Amazon ARIPL_718288
4,2023-05-31,Amazon ARIPL,718288,SAFF GOLD,8.050,11.08187,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,Amazon ARIPL_718288


In [397]:
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0

In [398]:
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])


In [399]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    offtake_df[['key', 'month_date', 'vol_in_rum']],
    on=['month_date', 'key'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [400]:
trend_file_df.select_dtypes('number').isna().sum()

pred_SARIMA                       1428
pred_p3m                             0
pred_p6m                             0
pred_prophet                     90371
pred_rf                              0
pred_value_SARIMA                 1428
pred_value_p3m                       0
pred_value_p6m                       0
pred_value_prophet               90371
pred_value_rf                        0
parent_material_code                 0
big_billion_days                 94190
big_billion_days_lag_1           94190
big_billion_days_lag_2           94190
big_billion_days_lead_1          94190
big_billion_days_lead_2          94190
ratio_last_year                  33347
quarter                              0
qtr_ind_rate                         0
vol_in_rum_value                     0
vol_in_rum_treated                   0
vol_in_rum_value_treated             0
cov                                  0
great_indian_festival           306785
great_indian_festival_lag_1     306785
great_indian_festival_lag

In [401]:
trend_file_df.select_dtypes('number').min().round()

pred_SARIMA                     -46917.0
pred_p3m                             0.0
pred_p6m                             0.0
pred_prophet                         0.0
pred_rf                              0.0
pred_value_SARIMA                   -1.0
pred_value_p3m                       0.0
pred_value_p6m                       0.0
pred_value_prophet                   0.0
pred_value_rf                        0.0
parent_material_code            709538.0
big_billion_days                     0.0
big_billion_days_lag_1               0.0
big_billion_days_lag_2               0.0
big_billion_days_lead_1              0.0
big_billion_days_lead_2              0.0
ratio_last_year                      0.0
quarter                              1.0
qtr_ind_rate                         0.0
vol_in_rum_value                     0.0
vol_in_rum_treated                   0.0
vol_in_rum_value_treated             0.0
cov                                  0.0
great_indian_festival                0.0
great_indian_fes

In [402]:
trend_file_df['vol_in_rum'].fillna(0, inplace=True)

In [403]:
for col in [ 'pred_best_model', 'pred_value_best_model', 'pred_prophet_70%ile','pred_prophet_60%ile', 'vol_in_rum']:
    trend_file_df[col] = trend_file_df[col].clip(lower=0)

In [404]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,pred_best_model,pred_value_best_model,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum
0,Big Basket_715100,2023-01-31,0.000000,0.5,0.683333,0.322004,0.326167,0.000000,0.000061,0.000083,0.000039,0.000040,715100,Big Basket,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.000000,1,1220.081000,0.000049,0.4,0.000049,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,None,Skin Care,0.425483,0.379470,0.4
1,Big Basket_715100,2023-02-28,0.400000,0.5,0.683333,0.483122,0.386167,0.000049,0.000061,0.000083,0.000059,0.000047,715100,Big Basket,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.000000,1,1220.081000,0.000049,0.4,0.000049,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,None,Skin Care,0.591834,0.536047,0.4
2,Big Basket_715100,2023-03-31,0.400000,0.5,0.683333,0.392078,0.652333,0.000049,0.000061,0.000083,0.000048,0.000080,715100,Big Basket,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.583333,1,1220.081000,0.000085,0.7,0.000085,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,None,Skin Care,0.512430,0.459295,0.7
3,Big Basket_715100,2023-04-30,0.517081,0.5,0.683333,0.520934,0.632333,0.000063,0.000061,0.000083,0.000064,0.000077,715100,Big Basket,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,0.875000,2,1220.081000,0.000085,0.7,0.000085,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,None,Skin Care,0.633398,0.582101,0.7
4,Big Basket_715100,2023-05-31,0.578341,0.6,0.683333,0.475287,0.774000,0.000071,0.000073,0.000083,0.000058,0.000094,715100,Big Basket,CO_SO_PCP,0.0,0.0,0.0,0.0,0.0,2.400000,2,1220.081000,0.000146,1.2,0.000146,2026-03-31,1.957888,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,None,Skin Care,0.586317,0.519760,1.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
400970,Nykaa_810605,2026-05-31,0.000000,0.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,810605,Nykaa,KAYA_ML,0.0,0.0,0.0,0.0,0.0,0.000000,2,1226.374229,0.000000,0.0,0.000000,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28,M+3,Skin Care,NaN,NaN,0.0
400971,Nykaa_810605,2026-06-30,0.000000,0.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,810605,Nykaa,KAYA_ML,0.0,0.0,0.0,0.0,0.0,0.000000,2,1226.374229,0.000000,0.0,0.000000,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28,M+4,Skin Care,NaN,NaN,0.0
400972,Nykaa_810605,2026-07-31,0.000000,0.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,810605,Nykaa,KAYA_ML,0.0,0.0,0.0,0.0,0.0,NaN,3,1226.374229,0.000000,0.0,0.000000,2026-01-31,1.734994,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-28,M+5,Skin Care,NaN,NaN,0.0
400973,Nykaa_810605,2026-08-31,0.000000,0.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,810605,Nykaa,KAYA_ML,0.0,0.0,0.0,0.0,0.0,NaN,3,1226.374229,0.000000,0.0,0.000000,2026-01-31,1.734994,run,acuuracy_check,/

In [405]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [406]:
trend_file_df['P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

trend_file_df['LY P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['LY P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [407]:
trend_file_df['LY P3M_copy'] = trend_file_df['LY P3M'].copy()

In [408]:
trend_file_df['P3M Max'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()

In [409]:
trend_file_df['P3M Top 2 Mean'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False

In [410]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [411]:
trend_file_df['MoM P3M growth'] = (
    trend_file_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)

In [412]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['MoM P3M growth_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
trend_file_df['MoM P3M growth_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)



In [413]:
trend_file_df['>=20%_3M_inc_month_count'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 

In [414]:
trend_file_df['Avg(P3M Mean, Max)'] = trend_file_df[['P3M', 'P3M Max']].mean(axis=1)

In [415]:
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
#         trend_file_df[col] = trend_file_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )

In [416]:
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    trend_file_df[f'{col}_value'] = trend_file_df[col] * trend_file_df['qtr_ind_rate'] / (10 ** 7)

In [417]:
trend_file_df['vol_in_rum_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['vol_in_rum'] / (10 ** 7)
trend_file_df['pred_prophet_70%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_70%ile'] / (10 ** 7)
trend_file_df['pred_prophet_60%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_60%ile'] / (10 ** 7)

In [418]:
value_cols = [col for col in trend_file_df.columns if 'value' in col]
value_cols

['pred_value_SARIMA',
 'pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'vol_in_rum_value',
 'vol_in_rum_value_treated',
 'pred_value_best_model',
 'P3M_value',
 'P6M_value',
 'LY P3M_value',
 'LY P6M_value',
 'pred_prophet_70%ile_value',
 'pred_prophet_60%ile_value']

In [419]:
for col in value_cols:
    try:
        assert trend_file_df[col].min() >= 0
    except:
        print(col)
    

    # trend_file_df[col] = trend_file_df[col] / (10 ** 7)

pred_value_SARIMA


In [420]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,pred_best_model,pred_value_best_model,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value
190320,Amazon RK_718472,2023-11-30,0.405,0.51,1.125,NaN,0.225,0.00002,0.000025,0.000056,NaN,0.000011,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,0.000000,4,496.828458,0.000022,0.45,0.000022,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,1.0,0.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190321,Amazon RK_718472,2023-12-31,0.405,0.51,1.125,NaN,0.090,0.00002,0.000025,0.000056,NaN,0.000004,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,NaN,4,496.828458,0.000000,0.00,0.000000,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,1.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190322,Amazon RK_718472,2024-01-31,0.405,0.51,1.125,NaN,0.855,0.00002,0.000025,0.000056,NaN,0.000042,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,2.400000,1,496.828458,0.000054,1.08,0.000054,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190323,Amazon RK_718472,2024-02-29,0.405,0.51,1.125,NaN,1.296,0.00002,0.000025,0.000056,NaN,0.000064,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,7.200000,1,496.828458,0.000080,1.62,0.000080,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,1.62,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,NaN,NaN
190324,Amazon RK_718472,2024-03-31,0.405,0.90,1.125,NaN,1.944,0.00002,0.000045,0.000056,NaN,0.000097,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,5.294118,1,496.828458,0.000134,2.70,0.000134,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,2.70,0.90,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6834,Big Basket_719192,2026-07-31,0.000,0.00,0.000,NaN,0.000,0.00000,0.000000,0.000000,NaN,0.000000,719192,Big Basket,VEG_CLEAN,0.0,0.0,0.0,0.0,0.0,0.000000,3,100.000000,0.000000,0.00,0.000000,2026-03-31,4.483165,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,M+3,Health & Hygiene,NaN,NaN,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,-100.000000,-100.0,-100.0,0.0,0.00,0.000000,0.0,0.0,0.0,NaN,NaN
6835,Big Basket_719192,2026-08-31,0.000,0.00,0.000,NaN,0.000,0.00000,0.000000,0.000000,NaN,0.000000,719192,Big Basket,VEG_CLEAN,0.0,0.0,0.0,0.0,0.0,0.000000,3,100.000000,0.000000,0.00,0.000000,2026-03-31,4.483165,run,acuuracy_check,

In [421]:
assert trend_file_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0

In [422]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['LY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

trend_file_df['LLY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


trend_file_df['LY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

trend_file_df['LLY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


trend_file_df['OT_Value_in_Cr_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

trend_file_df['OT_Value_in_Cr_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

trend_file_df['OT_Value_in_Cr_lag_3'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)

In [423]:
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [424]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'platform_name', 'brand_code',
       'big_billion_days', 'big_billion_days_lag_1', 'big_billion_days_lag_2',
       'big_billion_days_lead_1', 'big_billion_days_lead_2', 'ratio_last_year',
       'quarter', 'qtr_ind_rate', 'vol_in_rum_value', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'great_indian_festival', 'great_indian_festival_lag_1',
       'great_indian_festival_lag_2', 'great_indian_festival_lead_1',
       'great_indian_festival_lead_2', 'pred_best_model',
       'pred_value_best_model', 'run_month', 'M month', 'portfolio',
       'pred_prophet_70%ile', 'pred_prophet_60%ile', 'vol_in_rum', 'P3M',
       'P6M', 'LY P3M', 'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean',
   

In [425]:
# trend_file_df[['ASM', 'Depot', 'PSKU']] = trend_file_df['key'].str.split('_', expand=True)

In [426]:
trend_file_df.reset_index(drop=True, inplace=True)

In [427]:
trend_file_df.shape

(400975, 69)

In [428]:
trend_file_df['key'].nunique()

2509

In [429]:
# batch_info = pd.read_excel(
#     '/data/aniket/az_demand_forecasting-mil-sc/channel_wise_batch.xlsx'
# )

In [430]:
# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()

In [431]:
# brand_class_df.columns = ['brand_code', 'class']


In [432]:
# len_before_merge = len(trend_file_df)
# trend_file_df = trend_file_df.merge(
#     brand_class_df, 
#     on=['brand_code'],
#     how='left'
# )
# assert len_before_merge == len(trend_file_df)
# del len_before_merge

In [433]:
# trend_file_df['class'].isna().sum()

In [434]:
# trend_file_df['class'].unique()

missing combinations

In [435]:
# model_file = pd.read_csv("/data/aman_singh/acuuracy_check/Heuristic_QCOM_Chain_PSKU_Offtakes_live2.csv")
# model_file

In [436]:
model_file = trend_file_df.copy()

In [437]:
model_file['key'].nunique()

2509

In [438]:
run_month

Timestamp('2026-02-28 00:00:00')

In [439]:
data_query = f"""
    select * from TRN_DF_ECOM_OFFTAKE_CHAIN_PSKU_BACKTEST
    where month_date >= '2023-01-01'
        
"""
qcom_df = pd.read_sql(data_query, dev_conn)
qcom_df.head()

,MONTH_DATE,PLATFORM_NAME,PARENT_MATERIAL_CODE,BRAND_CODE,VOL_IN_RUM,INDEXBPM,IMPUTED,BIG_BILLION_DAYS,BIG_BILLION_DAYS_LAG_1,BIG_BILLION_DAYS_LAG_2,BIG_BILLION_DAYS_LEAD_1,BIG_BILLION_DAYS_LEAD_2,GREAT_INDIAN_FESTIVAL,GREAT_INDIAN_FESTIVAL_LAG_1,GREAT_INDIAN_FESTIVAL_LAG_2,GREAT_INDIAN_FESTIVAL_LEAD_1,GREAT_INDIAN_FESTIVAL_LEAD_2,RATIO_LAST_YEAR,QUARTER,RUN_MONTH
0,2023-01-31,Amazon ARIPL,718288,SAFF GOLD,9.640,13.27071,0,0,0,0,0,0,0,0,0,0,0,0.994832,1,2026-01-31
1,2023-02-28,Amazon ARIPL,718288,SAFF GOLD,7.915,10.89602,0,0,0,0,0,0,0,0,0,0,0,0.861631,1,2026-01-31
2,2023-03-31,Amazon ARIPL,718288,SAFF GOLD,9.505,13.08486,0,0,0,0,0,0,0,0,0,0,0,1.177605,1,2026-01-31
3,2023-04-30,Amazon ARIPL,718288,SAFF GOLD,9.290,12.78889,0,0,0,0,0,0,0,0,0,0,0,0.834181,2,2026-01-31
4,2023-05-31,Amazon ARIPL,718288,SAFF GOLD,8.050,11.08187,0,0,0,0,0,0,0,0,0,0,0,0.957359,2,2026-01-31


In [440]:
qcom_df.columns = qcom_df.columns.str.lower()

In [441]:
qcom_df['key'] = qcom_df[['platform_name', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [442]:
model_file['run_month'] = pd.to_datetime(model_file['run_month'])
model_file['month_date'] = pd.to_datetime(model_file['month_date'])

qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month_date'] = pd.to_datetime(qcom_df['month_date'])

In [443]:
tmp_df = model_file.groupby(['key', 'run_month'])['LY'].count().reset_index()
tmp_df#.isnull().sum()
#qcom_df[~qcom_df['key'].isin(model_file['key'].unique())]
qcom_df = qcom_df.merge(tmp_df, on = ['key', 'run_month'], how = 'left')
missing_df = qcom_df[qcom_df['LY'].isna()]
missing_df


,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,indexbpm,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,run_month,key,LY
529,2025-08-31,Amazon ARIPL,718494,SFOATS-FL,0.001140,0.00335,0,0,0,0,1,1,0,0,0,1,1,0.000000,3,2026-01-31,Amazon ARIPL_718494,NaN
530,2025-09-30,Amazon ARIPL,718494,SFOATS-FL,0.005852,0.01719,0,1,0,0,1,0,1,0,0,1,0,0.000000,3,2026-01-31,Amazon ARIPL_718494,NaN
531,2025-10-31,Amazon ARIPL,718494,SFOATS-FL,0.004940,0.01451,0,1,1,0,0,0,1,1,0,0,0,4.333336,4,2026-01-31,Amazon ARIPL_718494,NaN
532,2025-11-30,Amazon ARIPL,718494,SFOATS-FL,0.025498,0.07489,0,0,1,1,0,0,0,1,1,0,0,7.293484,4,2026-01-31,Amazon ARIPL_718494,NaN
533,2025-12-31,Amazon ARIPL,718494,SFOATS-FL,0.033402,0.09811,0,0,0,1,0,0,0,0,1,0,0,8.398094,4,2026-01-31,Amazon ARIPL_718494,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
466885,2026-09-30,Nykaa,811019,PADV_WIPS,0.000000,0.00000,1,0,0,0,0,0,0,0,0,0,0,0.000000,3,2026-04-30,Nykaa_811019,NaN
466886,2026-10-31,Nykaa,811019,PADV_WIPS,0.000000,0.00000,1,0,0,0,0,0,0,0,0,0,0,NaN,4,2026-04-30,Nykaa_811019,NaN
466887,2026-11-30,Nykaa,811019,PADV_WIPS,0.000000,0.00000,1,0,0,0,0,0,0,0,0,0,0,NaN,4,2026-04-30,Nykaa_811019,NaN
466888,2026-12-31,Nykaa,811019,PADV_WIPS,0.000000,0.00000,1,0,0,0,0,0,0,0,0,0,0,NaN,4,2026-04-30,Nykaa_811019,NaN


In [444]:
missing_df['key'].nunique()

767

In [445]:
missing_df = missing_df[['key','run_month','month_date', 'platform_name', 'parent_material_code', 'brand_code',
       'vol_in_rum']]
missing_df

,key,run_month,month_date,platform_name,parent_material_code,brand_code,vol_in_rum
529,Amazon ARIPL_718494,2026-01-31,2025-08-31,Amazon ARIPL,718494,SFOATS-FL,0.001140
530,Amazon ARIPL_718494,2026-01-31,2025-09-30,Amazon ARIPL,718494,SFOATS-FL,0.005852
531,Amazon ARIPL_718494,2026-01-31,2025-10-31,Amazon ARIPL,718494,SFOATS-FL,0.004940
532,Amazon ARIPL_718494,2026-01-31,2025-11-30,Amazon ARIPL,718494,SFOATS-FL,0.025498
533,Amazon ARIPL_718494,2026-01-31,2025-12-31,Amazon ARIPL,718494,SFOATS-FL,0.033402
...,...,...,...,...,...,...,...
466885,Nykaa_811019,2026-04-30,2026-09-30,Nykaa,811019,PADV_WIPS,0.000000
466886,Nykaa_811019,2026-04-30,2026-10-31,Nykaa,811019,PADV_WIPS,0.000000
466887,Nykaa_811019,2026-04-30,2026-11-30,Nykaa,811019,PADV_WIPS,0.000000
466888,Nykaa_811019,2026-04-30,2026-12-31,Nykaa,811019,PADV_WIPS,0.000000


In [446]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


Credentials retrieved successfully for prod db.


,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [447]:
len_before_merge = len(missing_df)

missing_df = missing_df.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(missing_df)

In [448]:
missing_df

,key,run_month,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,qtr_ind_rate
0,Amazon ARIPL_718494,2026-01-31,2025-08-31,Amazon ARIPL,718494,SFOATS-FL,0.001140,292663.137458
1,Amazon ARIPL_718494,2026-01-31,2025-09-30,Amazon ARIPL,718494,SFOATS-FL,0.005852,292663.137458
2,Amazon ARIPL_718494,2026-01-31,2025-10-31,Amazon ARIPL,718494,SFOATS-FL,0.004940,292663.137458
3,Amazon ARIPL_718494,2026-01-31,2025-11-30,Amazon ARIPL,718494,SFOATS-FL,0.025498,292663.137458
4,Amazon ARIPL_718494,2026-01-31,2025-12-31,Amazon ARIPL,718494,SFOATS-FL,0.033402,292663.137458
...,...,...,...,...,...,...,...,...
46058,Nykaa_811019,2026-04-30,2026-09-30,Nykaa,811019,PADV_WIPS,0.000000,366.484998
46059,Nykaa_811019,2026-04-30,2026-10-31,Nykaa,811019,PADV_WIPS,0.000000,366.484998
46060,Nykaa_811019,2026-04-30,2026-11-30,Nykaa,811019,PADV_WIPS,0.000000,366.484998
46061,Nykaa_811019,2026-04-30,2026-12-31,Nykaa,811019,PADV_WIPS,0.000000,366.484998


In [449]:
missing_df['month_date'] = pd.to_datetime(missing_df['month_date'])
missing_df['run_month'] = pd.to_datetime(missing_df['run_month'])


mappings = {}

for run_month in missing_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 11):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   
missing_df['M month'] = missing_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(missing_df)

# Merge 70th percentile Prophet predictions

assert missing_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0
missing_df.drop('vol_in_rum', axis=1, inplace=True)

In [450]:
missing_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7', 'M+8',
       'M+9'], dtype=object)

In [451]:
offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    offtake_df[['key', 'month_date', 'vol_in_rum']],
    on=['month_date', 'key'],
    how='left'
)
assert len(missing_df) == len_before_merge

missing_df['vol_in_rum'].fillna(0, inplace=True)
for col in [ 'vol_in_rum']:
    missing_df[col] = missing_df[col].clip(lower=0)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [452]:
missing_df['P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=1).mean()

missing_df['P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=3).mean()

missing_df['LY P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

missing_df['LY P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [453]:

missing_df['LY P3M_copy'] = missing_df['LY P3M'].copy()
missing_df['P3M Max'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()
missing_df['P3M Top 2 Mean'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth'] = (
    missing_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
missing_df['MoM P3M growth_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)


missing_df['>=20%_3M_inc_month_count'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 
missing_df['Avg(P3M Mean, Max)'] = missing_df[['P3M', 'P3M Max']].mean(axis=1)
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
#         missing_df[col] = missing_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )
# missing_df.to_csv('collate_check.csv', index=False)
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    missing_df[f'{col}_value'] = missing_df[col] * missing_df['qtr_ind_rate'] / (10 ** 7)
missing_df['vol_in_rum_value'] = missing_df['qtr_ind_rate'] * missing_df['vol_in_rum'] / (10 ** 7)
value_cols = [col for col in missing_df.columns if 'value' in col]
value_cols
for col in value_cols:
    try:
        assert missing_df[col].min() >= 0
    except:
        print(col)
    

    # missing_df[col] = missing_df[col] / (10 ** 7)
missing_df
assert missing_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['LY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

missing_df['LLY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


missing_df['LY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

missing_df['LLY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


missing_df['OT_Value_in_Cr_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

missing_df['OT_Value_in_Cr_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

missing_df['OT_Value_in_Cr_lag_3'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

missing_df.reset_index(drop=True, inplace=True)

# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()
# brand_class_df.columns = ['brand_code', 'class']

# len_before_merge = len(missing_df)
# missing_df = missing_df.merge(
#     brand_class_df, 
#     on=['brand_code'],
#     how='left'
# )
# assert len_before_merge == len(missing_df)
# del len_before_merge
# missing_df['class'].isna().sum()
# missing_df['class'].unique()

LY P3M_value


In [454]:
pd.set_option('display.max_columns', None)

In [455]:
# missing_df[
#     # (missing_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (missing_df['month_date'] > '2024-06-30') &
#     (missing_df['M month'].notna())
#     # (missing_df['class'].isin(['B', 'C']))
# ].to_csv('missing_combinations_QCOM_Chain_city_PSKU_Offtakes.csv', index=False)

In [456]:
model_file

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,pred_best_model,pred_value_best_model,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3
0,Amazon RK_718472,2023-11-30,0.405,0.51,1.125,NaN,0.225,0.00002,0.000025,0.000056,NaN,0.000011,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,0.000000,4,496.828458,0.000022,0.45,0.000022,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,1.0,0.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazon RK_718472,2023-12-31,0.405,0.51,1.125,NaN,0.090,0.00002,0.000025,0.000056,NaN,0.000004,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,NaN,4,496.828458,0.000000,0.00,0.000000,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,1.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000022,NaN,NaN
2,Amazon RK_718472,2024-01-31,0.405,0.51,1.125,NaN,0.855,0.00002,0.000025,0.000056,NaN,0.000042,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,2.400000,1,496.828458,0.000054,1.08,0.000054,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000022,NaN
3,Amazon RK_718472,2024-02-29,0.405,0.51,1.125,NaN,1.296,0.00002,0.000025,0.000056,NaN,0.000064,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,7.200000,1,496.828458,0.000080,1.62,0.000080,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,1.62,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000054,0.000000,0.000022
4,Amazon RK_718472,2024-03-31,0.405,0.90,1.125,NaN,1.944,0.00002,0.000045,0.000056,NaN,0.000097,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,5.294118,1,496.828458,0.000134,2.70,0.000134,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,2.70,0.90,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000080,0.000054,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
400970,Big Basket_719192,2026-07-31,0.000,0.00,0.000,NaN,0.000,0.00000,0.000000,0.000000,NaN,0.000000,719192,Big Basket,VEG_CLEAN,0.0,0.0,0.0,0.0,0.0,0.000000,3,100.000000,0.000000,0.00,0.000000,2026-03-31,4.483165,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,M+3,Health & Hygiene,NaN,NaN,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,-100.000000,-100.

In [457]:
model_file['skipped'] = 0
missing_df['skipped'] = 1
final_df = pd.concat([model_file,missing_df])
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,pred_best_model,pred_value_best_model,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,Amazon RK_718472,2023-11-30,0.405,0.51,1.125,NaN,0.225,0.00002,0.000025,0.000056,NaN,0.000011,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,0.000000,4.0,496.828458,0.000022,0.45,0.000022,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,1.0,0.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,Amazon RK_718472,2023-12-31,0.405,0.51,1.125,NaN,0.090,0.00002,0.000025,0.000056,NaN,0.000004,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,NaN,4.0,496.828458,0.000000,0.00,0.000000,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,1.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000022,NaN,NaN,0
2,Amazon RK_718472,2024-01-31,0.405,0.51,1.125,NaN,0.855,0.00002,0.000025,0.000056,NaN,0.000042,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,2.400000,1.0,496.828458,0.000054,1.08,0.000054,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000022,NaN,0
3,Amazon RK_718472,2024-02-29,0.405,0.51,1.125,NaN,1.296,0.00002,0.000025,0.000056,NaN,0.000064,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,7.200000,1.0,496.828458,0.000080,1.62,0.000080,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,1.62,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000054,0.000000,0.000022,0
4,Amazon RK_718472,2024-03-31,0.405,0.90,1.125,NaN,1.944,0.00002,0.000045,0.000056,NaN,0.000097,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,5.294118,1.0,496.828458,0.000134,2.70,0.000134,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,2026-01-31,None,Hair Oils,NaN,NaN,2.70,0.90,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000080,0.000054,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46058,Myntra_811169,2026-09-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,811169,Myntra,SW_SGPRF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1712.605337,0.000000,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-30,M+5,Male Grooming,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
46059,Myntra_811169,2026-10-

In [458]:
final_df[final_df.select_dtypes(include='number').columns] = final_df.select_dtypes(include='number').fillna(0)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,pred_best_model,pred_value_best_model,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,Amazon RK_718472,2023-11-30,0.405,0.51,1.125,0.0,0.225,0.00002,0.000025,0.000056,0.0,0.000011,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.000000,4.0,496.828458,0.000022,0.45,0.000022,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0
1,Amazon RK_718472,2023-12-31,0.405,0.51,1.125,0.0,0.090,0.00002,0.000025,0.000056,0.0,0.000004,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.000000,4.0,496.828458,0.000000,0.00,0.000000,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.000000,0
2,Amazon RK_718472,2024-01-31,0.405,0.51,1.125,0.0,0.855,0.00002,0.000025,0.000056,0.0,0.000042,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.400000,1.0,496.828458,0.000054,1.08,0.000054,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.000000,0
3,Amazon RK_718472,2024-02-29,0.405,0.51,1.125,0.0,1.296,0.00002,0.000025,0.000056,0.0,0.000064,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,7.200000,1.0,496.828458,0.000080,1.62,0.000080,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,1.62,0.51,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.51,0.000025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000054,0.000000,0.000022,0
4,Amazon RK_718472,2024-03-31,0.405,0.90,1.125,0.0,1.944,0.00002,0.000045,0.000056,0.0,0.000097,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,5.294118,1.0,496.828458,0.000134,2.70,0.000134,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,2.70,0.90,0.0,0.0,0.0,0.0,0.0,0.0,76.470588,0.0,0.0,0.0,0.90,0.000045,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000080,0.000054,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46058,Myntra_811169,2026-09-30,0.000,0.00,0.000,0.0,0.000,0.00000,0.000000,0.000000,0.0,0.000000,811169,Myntra,SW_SGPRF,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1712.605337,0.000000,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-04-30,M+5,Male Grooming,0.0,0.0,0.00,0.00,0.0,0.0

In [459]:
final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,pred_best_model,pred_value_best_model,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,Amazon RK_718472,2023-11-30,0.405,0.51,1.125,0.0,0.225,0.00002,0.000025,0.000056,0.0,0.000011,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.000000,4.0,496.828458,0.000022,0.45,0.000022,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0
1,Amazon RK_718472,2023-12-31,0.405,0.51,1.125,0.0,0.090,0.00002,0.000025,0.000056,0.0,0.000004,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.000000,4.0,496.828458,0.000000,0.00,0.000000,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.000000,0
2,Amazon RK_718472,2024-01-31,0.405,0.51,1.125,0.0,0.855,0.00002,0.000025,0.000056,0.0,0.000042,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.400000,1.0,496.828458,0.000054,1.08,0.000054,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.000000,0
3,Amazon RK_718472,2024-02-29,0.405,0.51,1.125,0.0,1.296,0.00002,0.000025,0.000056,0.0,0.000064,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,7.200000,1.0,496.828458,0.000080,1.62,0.000080,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,1.62,0.51,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.51,0.000025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000054,0.000000,0.000022,0
4,Amazon RK_718472,2024-03-31,0.405,0.90,1.125,0.0,1.944,0.00002,0.000045,0.000056,0.0,0.000097,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,5.294118,1.0,496.828458,0.000134,2.70,0.000134,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,2.70,0.90,0.0,0.0,0.0,0.0,0.0,0.0,76.470588,0.0,0.0,0.0,0.90,0.000045,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000080,0.000054,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46058,Myntra_811169,2026-09-30,0.000,0.00,0.000,0.0,0.000,0.00000,0.000000,0.000000,0.0,0.000000,811169,Myntra,SW_SGPRF,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1712.605337,0.000000,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-04-30,M+5,Male Grooming,0.0,0.0,0.00,0.00,0.0,0.0

Heuristic new approac

In [460]:
# pip install pymannkendall

identify events

In [461]:
final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['run_month'] = pd.to_datetime(final_df['run_month'])

In [462]:
import numpy as np 
import pandas as pd 
EVENT_MONTHS = [9, 10, 11] # Sep, Oct, Nov
def detect_event_months(df_grp):
    df_grp = df_grp.sort_values("month_date").copy()
    run_month = df_grp["run_month"].max()

    # only historical data
    hist = df_grp[df_grp["month_date"] < run_month].copy()

    # initialize
    df_grp["event_month_flag"] = 0
    df_grp["event_uplift_factor"] = 0.0

    if len(hist) < 12:
        df_grp["event_sensitive_flag"] = 0
        return df_grp

    event_sensitive = 0

    # loop year-wise
    for year in hist["month_date"].dt.year.unique():

        year_df = hist[hist["month_date"].dt.year == year]

        for _, row in year_df.iterrows():

            month = row["month_date"].month

            if month not in EVENT_MONTHS:
                continue

            curr_date = row["month_date"]

            # previous 12 months before this month
            prev_12m = hist[
                (hist["month_date"] < curr_date) &
                (hist["month_date"] >= curr_date - pd.DateOffset(months=12)) &
                (~hist["month_date"].dt.month.isin(EVENT_MONTHS))  # 
            ]

            if len(prev_12m) < 6:
                continue

            prev_12m_avg = prev_12m["vol_in_rum_value"].mean()

            if prev_12m_avg <= 0 or np.isnan(prev_12m_avg):
                continue

            uplift = row["vol_in_rum_value"] / prev_12m_avg

            if uplift > 2:
                mask = df_grp["month_date"] == curr_date
                df_grp.loc[mask, "event_month_flag"] = 1
                df_grp.loc[mask, "event_uplift_factor"] = uplift
                event_sensitive = 1

    df_grp["event_sensitive_flag"] = event_sensitive
    return df_grp


In [463]:
final_df = (
    final_df
    .groupby(
        ["platform_name", "parent_material_code", "run_month"],
        group_keys=False
    )
    .apply(detect_event_months)
)


In [464]:
# final_df[(final_df['event_month_flag'] == 1) & (final_df['platform_name'] == 'Flipkart National') & ((final_df['parent_material_code'] == 718939))]

In [465]:
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["event_month_flag"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()


In [466]:
adj_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_adj": compute_adjusted_pm(x, 3,'vol_in_rum',0),
        "P6M_adj": compute_adjusted_pm(x, 6,'vol_in_rum',0),
        "P3M_adj_value": compute_adjusted_pm(x, 3,'vol_in_rum_value',0),
        "P6M_adj_value": compute_adjusted_pm(x, 6,'vol_in_rum_value',0)
    })
).reset_index()

final_df = final_df.merge(
    adj_df,
    on=["platform_name", "parent_material_code", "run_month"],
    how="left"
)

In [467]:
adj_ly_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_adj": compute_adjusted_pm(x, 3,'vol_in_rum',year_shift=1),
        "LY_P6M_adj": compute_adjusted_pm(x, 6,'vol_in_rum', year_shift=1),
        "LY_P3M_adj_value": compute_adjusted_pm(x, 3,'vol_in_rum_value', year_shift=1),
        "LY_P6M_adj_value": compute_adjusted_pm(x, 6,"vol_in_rum_value", year_shift=1)
    })
).reset_index()

In [468]:
final_df = final_df.merge(
    adj_ly_df,
    on=["platform_name", "parent_material_code", "run_month"],
    how="left"
)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,pred_best_model,pred_value_best_model,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value
0,Amazon RK_718472,2023-11-30,0.405,0.51,1.125,0.0,0.225,0.00002,0.000025,0.000056,0.0,0.000011,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000022,0.45,0.000022,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.63,0.0,0.000031
1,Meesho_718472,2025-12-31,0.000,0.00,0.000,0.0,0.000,0.00000,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,42.66,42.66,0.002119,0.002119,NaN,NaN,NaN,NaN
2,Amazon RK_718472,2023-12-31,0.405,0.51,1.125,0.0,0.090,0.00002,0.000025,0.000056,0.0,0.000004,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000000,0.00,0.000000,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.63,0.0,0.000031
3,Meesho_718472,2026-01-31,0.000,0.00,0.000,0.0,0.000,0.00000,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,M,Hair Oils,0.0,0.0,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,42.66,42.66,0.002119,0.002119,NaN,NaN,NaN,NaN
4,Amazon RK_718472,2024-01-31,0.405,0.51,1.125,0.0,0.855,0.00002,0.000025,0.000056,0.0,0.000042,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.4,1.0,496.828458,0.000054,1.08,0.000054,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.63,0.0,0.000031
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447033,Big Basket_719192,20

In [469]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


len_before_merge = len(final_df)

final_df = final_df.merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(final_df)


Credentials retrieved successfully for prod db.


In [470]:
import pandas as pd
import numpy as np
import pymannkendall as mk

def detect_trend_for_group(df_grp):
    """
    Detect final trend flag and p3m_slope_flag separately.
    Must contain 'month_date', 'vol_in_rum', 'run_month'
    """

    # ---------- 1. Sort ----------
    df_grp = df_grp.sort_values("month_date")

    # ---------- 2. Identify run_month ----------
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]

    # if no actual data → no trend
    if df_actual.empty or len(df_actual) < 4:
        return pd.Series({"trend_flag": 0, "p3m_slope_flag": 0})

    # ---------- 3. MK Trend ----------
    series = df_actual["vol_in_rum_value"].astype(float)

    try:
        mk_result = mk.original_test(series)
        if mk_result.trend == "increasing":
            mk_trend = 1
        elif mk_result.trend == "decreasing":
            mk_trend = -1
        else:
            mk_trend = 0
    except:
        mk_trend = 0

    # ---------- 4. P3M Slope ----------
    # last 4 months → take last 3 with shift
    #shifted_series = series.shift(1).dropna()

    p3m_values = series.tail(3).values
    #print(p3m_values)

    if len(p3m_values) < 3:
        slope_flag = 0
    else:
        x = np.arange(3)
        slope = np.polyfit(x, p3m_values, 1)[0]
        slope_flag = 1 if slope > 0 else (-1 if slope < 0 else 0)
        

    return pd.Series({
        "trend_flag": mk_trend,
        "p3m_slope_flag": slope_flag
    })


# ---------------------------------------------------------
# APPLY ON ENTIRE DATASET
# ---------------------------------------------------------

# trend_df = final_df.groupby(
#     ["platform_name", "parent_material_code"]
# ).apply(detect_trend_for_group).reset_index()

# trend_df = final_df[final_df['key'] == 'Zepto_721898'].groupby(
#     ["platform_name", "parent_material_code", "run_month"]
# ).apply(detect_trend_for_group).reset_index()
trend_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(detect_trend_for_group).reset_index()

In [471]:
trend_df["final_trend"] = np.where(
    (trend_df["trend_flag"] == 1) & (trend_df["p3m_slope_flag"] == 1), 1,
    np.where(
        (trend_df["trend_flag"] == -1) & (trend_df["p3m_slope_flag"] == -1), -1,
        0
    )
)
trend_df

,platform_name,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend
0,Amazon ARIPL,718288,2026-01-31,1,1,1
1,Amazon ARIPL,718288,2026-02-28,1,1,1
2,Amazon ARIPL,718288,2026-03-31,1,-1,0
3,Amazon ARIPL,718288,2026-04-30,1,1,1
4,Amazon ARIPL,718321,2026-01-31,-1,0,0
...,...,...,...,...,...,...
12887,Nykaa,810805,2026-04-30,0,0,0
12888,Nykaa,811019,2026-01-31,0,0,0
12889,Nykaa,811019,2026-02-28,0,0,0
12890,Nykaa,811019,2026-03-31,0,0,0


## detect seasonality

In [472]:
from statsmodels.tsa.stattools import acf
import numpy as np
import pandas as pd

def detect_yearly_seasonality(df_grp, threshold=0.3):
    """
    Detects yearly seasonality using ACF at lag=12 only.
    Uses vol_in_rum as the metric.
    """
    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]
    series = df_actual["vol_in_rum"].astype(float).values

    # Need at least 18 points to compare last year vs this year
    if len(series) < 18:
        return 0

    # Compute ACF up to lag-12
    acf_vals = acf(series, nlags=12, fft=False)

    lag12_acf = acf_vals[12]

    # absolute ACF because seasonal correlation can be negative as well
    if abs(lag12_acf) >= threshold:
        return 1
    else:
        return 0
    

seasonality_df = final_df.groupby(
    ["platform_name", "brand_code", 'run_month']
).apply(detect_yearly_seasonality).reset_index(name="seasonality_flag")

seasonality_df



,platform_name,brand_code,run_month,seasonality_flag
0,Amazon ARIPL,CO_SO_VCN,2026-01-31,0
1,Amazon ARIPL,CO_SO_VCN,2026-02-28,0
2,Amazon ARIPL,CO_SO_VCN,2026-03-31,0
3,Amazon ARIPL,CO_SO_VCN,2026-04-30,0
4,Amazon ARIPL,SAF-MUSLI,2026-01-31,0
...,...,...,...,...
2335,Nykaa,SW_HR_WAX,2026-04-30,0
2336,Nykaa,SW_SGPRF,2026-01-31,0
2337,Nykaa,SW_SGPRF,2026-02-28,0
2338,Nykaa,SW_SGPRF,2026-03-31,0


In [473]:
# seasonality_df.to_csv('seasonal_ecom.csv')

In [474]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["vol_in_rum_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,platform_name,parent_material_code,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,Amazon ARIPL,718288,2026-01-31,0.126748,0.384173,0.229718,0.051485
1,Amazon ARIPL,718288,2026-02-28,0.130513,0.402740,0.239404,0.054445
2,Amazon ARIPL,718288,2026-03-31,0.147374,0.401072,0.248853,0.050740
3,Amazon ARIPL,718288,2026-04-30,0.093064,0.558584,0.279272,0.093104
4,Amazon ARIPL,718321,2026-01-31,0.000000,0.000000,0.000000,0.000000


In [475]:
trend_df = trend_df.merge(threshold_df, on = ['platform_name', 'parent_material_code', 'run_month'], how = 'left')
trend_df

,platform_name,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend,lower_threshold,upper_threshold,mean_value,std_value
0,Amazon ARIPL,718288,2026-01-31,1,1,1,0.126748,0.384173,0.229718,0.051485
1,Amazon ARIPL,718288,2026-02-28,1,1,1,0.130513,0.402740,0.239404,0.054445
2,Amazon ARIPL,718288,2026-03-31,1,-1,0,0.147374,0.401072,0.248853,0.050740
3,Amazon ARIPL,718288,2026-04-30,1,1,1,0.093064,0.558584,0.279272,0.093104
4,Amazon ARIPL,718321,2026-01-31,-1,0,0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
12887,Nykaa,810805,2026-04-30,0,0,0,0.000259,0.001035,0.000517,0.000113
12888,Nykaa,811019,2026-01-31,0,0,0,0.000000,0.000000,0.000000,0.000000
12889,Nykaa,811019,2026-02-28,0,0,0,0.000000,0.000000,0.000000,0.000000
12890,Nykaa,811019,2026-03-31,0,0,0,0.000000,0.000000,0.000000,0.000000


In [476]:
# trend_df.to_csv('t_s_t_df_ecom.csv')

In [477]:
final_df = final_df.merge(seasonality_df, on = ["platform_name", "brand_code", 'run_month'], how = 'left')
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,pred_best_model,pred_value_best_model,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag
0,Amazon RK_718472,2023-11-30,0.405,0.51,1.125,0.0,0.225,0.00002,0.000025,0.000056,0.0,0.000011,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000022,0.45,0.000022,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.63,0.0,0.000031,496.828458,0
1,Meesho_718472,2025-12-31,0.000,0.00,0.000,0.0,0.000,0.00000,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,42.66,42.66,0.002119,0.002119,NaN,NaN,NaN,NaN,496.828458,0
2,Amazon RK_718472,2023-12-31,0.405,0.51,1.125,0.0,0.090,0.00002,0.000025,0.000056,0.0,0.000004,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000000,0.00,0.000000,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.63,0.0,0.000031,496.828458,0
3,Meesho_718472,2026-01-31,0.000,0.00,0.000,0.0,0.000,0.00000,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,M,Hair Oils,0.0,0.0,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,42.66,42.66,0.002119,0.002119,NaN,NaN,NaN,NaN,496.828458,0
4,Amazon RK_718472,2024-01-31,0.405,0.51,1.125,0.0,0.855,0.00002,0.000025,0.000056,0.0,0.000042,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.4,1.0,496.828458,0.000054,1.08,0.000054,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.63,0.0,0.000031,496.828458,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

In [478]:
trend_df.columns

Index(['platform_name', 'parent_material_code', 'run_month', 'trend_flag',
       'p3m_slope_flag', 'final_trend', 'lower_threshold', 'upper_threshold',
       'mean_value', 'std_value'],
      dtype='object')

In [479]:
final_df = final_df.merge(trend_df[['platform_name', 'parent_material_code', 'run_month',
                                    'final_trend','lower_threshold', 'upper_threshold']], on = ["platform_name", "parent_material_code", 'run_month'], how = 'left')
final_df


,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,pred_best_model,pred_value_best_model,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,Amazon RK_718472,2023-11-30,0.405,0.51,1.125,0.0,0.225,0.00002,0.000025,0.000056,0.0,0.000011,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000022,0.45,0.000022,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.63,0.0,0.000031,496.828458,0,0,0.00000,0.000000
1,Meesho_718472,2025-12-31,0.000,0.00,0.000,0.0,0.000,0.00000,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,42.66,42.66,0.002119,0.002119,NaN,NaN,NaN,NaN,496.828458,0,0,0.00106,0.004239
2,Amazon RK_718472,2023-12-31,0.405,0.51,1.125,0.0,0.090,0.00002,0.000025,0.000056,0.0,0.000004,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000000,0.00,0.000000,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.63,0.0,0.000031,496.828458,0,0,0.00000,0.000000
3,Meesho_718472,2026-01-31,0.000,0.00,0.000,0.0,0.000,0.00000,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,M,Hair Oils,0.0,0.0,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,42.66,42.66,0.002119,0.002119,NaN,NaN,NaN,NaN,496.828458,0,0,0.00106,0.004239
4,Amazon RK_718472,2024-01-31,0.405,0.51,1.125,0.0,0.855,0.00002,0.000025,0.000056,0.0,0.000042,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.4,1.0,496.828458,0.000054,1.08,0.000054,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.63,0.0,0.000031,496.828458,0,0,0.00000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

In [480]:
final_df[final_df.select_dtypes(include='number').columns] = final_df.select_dtypes(include='number').fillna(0)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,pred_best_model,pred_value_best_model,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,Amazon RK_718472,2023-11-30,0.405,0.51,1.125,0.0,0.225,0.00002,0.000025,0.000056,0.0,0.000011,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000022,0.45,0.000022,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.63,0.0,0.000031,496.828458,0,0,0.00000,0.000000
1,Meesho_718472,2025-12-31,0.000,0.00,0.000,0.0,0.000,0.00000,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,42.66,42.66,0.002119,0.002119,0.0,0.00,0.0,0.000000,496.828458,0,0,0.00106,0.004239
2,Amazon RK_718472,2023-12-31,0.405,0.51,1.125,0.0,0.090,0.00002,0.000025,0.000056,0.0,0.000004,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,4.0,496.828458,0.000000,0.00,0.000000,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.63,0.0,0.000031,496.828458,0,0,0.00000,0.000000
3,Meesho_718472,2026-01-31,0.000,0.00,0.000,0.0,0.000,0.00000,0.000000,0.000000,0.0,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,M,Hair Oils,0.0,0.0,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,42.66,42.66,0.002119,0.002119,0.0,0.00,0.0,0.000000,496.828458,0,0,0.00106,0.004239
4,Amazon RK_718472,2024-01-31,0.405,0.51,1.125,0.0,0.855,0.00002,0.000025,0.000056,0.0,0.000042,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,2.4,1.0,496.828458,0.000054,1.08,0.000054,2025-12-31,2.322408,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-01-31,None,Hair Oils,0.0,0.0,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.000000,0.0,0.63,0.0,0.000031,496.828458,0,0,0.00000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

In [481]:
final_df.drop(columns = ['big_billion_days', 'big_billion_days_lag_1', 'big_billion_days_lag_2',
       'big_billion_days_lead_1', 'big_billion_days_lead_2', 'great_indian_festival',
       'great_indian_festival_lag_1', 'great_indian_festival_lag_2',
       'great_indian_festival_lead_1', 'great_indian_festival_lead_2'], inplace = True)

In [482]:
final_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'platform_name', 'brand_code',
       'ratio_last_year', 'quarter', 'qtr_ind_rate_x', 'vol_in_rum_value',
       'vol_in_rum_treated', 'vol_in_rum_value_treated', 'train_till', 'cov',
       'run', 'step', 'file_path', 'pred_best_model', 'pred_value_best_model',
       'run_month', 'M month', 'portfolio', 'pred_prophet_70%ile',
       'pred_prophet_60%ile', 'vol_in_rum', 'P3M', 'P6M', 'LY P3M', 'LY P6M',
       'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'pred_prophet_60%ile_value', 'LY', 'LLY',
       'LY value', 'LLY value', '

In [483]:
missing_df['LY P3M'].sum()

0.0

In [484]:
final_df = final_df.sort_values(['key', 'run_month','month_date'])

# base LY


# LY lags
final_df['ly_lag1_value'] = (
    final_df
    .groupby(['key','run_month'])['vol_in_rum_value']
    .shift(13)
)

final_df['ly_lag2_value'] = (
    final_df
    .groupby(['key', 'run_month'])['vol_in_rum_value']
    .shift(14)
)

# LY leads
final_df['ly_lead1_value'] = (
    final_df
    .groupby(['key', 'run_month'])['vol_in_rum_value']
    .shift(11)
)

final_df['ly_lead2_value'] = (
    final_df
    .groupby(['key', 'run_month'])['vol_in_rum_value']
    .shift(10)
)


In [485]:
final_df[final_df['month_date'] == '2026-04-30']['P3M_value'].sum()

169.45890814389134

In [486]:
brand_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'brand')
brand_seas.columns = brand_seas.columns.str.lower()
brand_seas.rename(columns={'brand':'brand_code', 'months_num':'month', 'flag':'is_seasonal_month'}, inplace=True)

final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['month'] = final_df['month_date'].dt.month
final_df = final_df.merge(brand_seas, on = ['brand_code', 'month'], how = 'left')
final_df['is_seasonal_month'].fillna(0, inplace=True)

psku_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'psku')
psku_seas.columns = psku_seas.columns.str.lower()
psku_seas.rename(columns={'months_num':'month', 'flag':'is_seasonal_month_psku'}, inplace=True)

final_df = final_df.merge(psku_seas[['parent_material_code', 'month','is_seasonal_month_psku']], on = ['parent_material_code', 'month'], how = 'left')
final_df['is_seasonal_month_psku'].fillna(0, inplace=True)
final_df['final_seasonal_month'] = np.where(
    (final_df['is_seasonal_month'] == 1) | (final_df['is_seasonal_month_psku'] == 1), 1, 0
)



In [487]:
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["final_seasonal_month"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()

adj_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',0),
        "P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum',0),
        "P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value',0),
        "P6M_non_seasonal_value": compute_adjusted_pm(x, 6,'vol_in_rum_value',0)
    })
).reset_index()
adj_df


adj_ly_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',year_shift=1),
        "LY_P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum', year_shift=1),
        "LY_P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value', year_shift=1),
        "LY_P6M_non_seasonal_value": compute_adjusted_pm(x, 6,"vol_in_rum_value", year_shift=1)
    })
).reset_index()

adj_df = adj_df.merge(adj_ly_df, on = ['key', 'run_month'], how = 'left')
#adj_df[adj_df['key'] == 'reliance_b2c_2_haryana_718488']
adj_df

,key,run_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,Amazon ARIPL_718288,2026-01-31,21.770,19.549,0.302310,0.271468,17.926,14.318833,0.248930,0.198839
1,Amazon ARIPL_718288,2026-02-28,22.136,20.106,0.307392,0.279202,15.386,15.291667,0.213658,0.212348
2,Amazon ARIPL_718288,2026-03-31,22.018,20.948,0.305754,0.290895,13.956,15.257833,0.193800,0.211878
3,Amazon ARIPL_718288,2026-04-30,27.264,24.517,0.378602,0.340456,12.990,15.458000,0.180386,0.214658
4,Amazon ARIPL_718321,2026-01-31,0.000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
12887,Nykaa_810805,2026-04-30,14.118,14.118,0.000517,0.000517,NaN,NaN,NaN,NaN
12888,Nykaa_811019,2026-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12889,Nykaa_811019,2026-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12890,Nykaa_811019,2026-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [488]:
final_df.shape

(447038, 85)

In [489]:
#adj_df.to_csv('seasonal_p3m_qcom.csv', index=False)
#all[all['month_date'].isin(['2025-11-30','2025-12-31','2026-01-31'])].groupby(['key','run_month','month_date','final_seasonal_month'])['vol_in_rum'].sum().reset_index().to_csv('seasonal_month_check.csv', index=False)
final_df = final_df.merge(
    adj_df,
    on=['key','run_month'],
    how="left"
)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,pred_best_model,pred_value_best_model,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,Amazon ARIPL_718288,2023-01-31,0.000000,9.020000,9.154167,9.883636,9.323507,0.000000,0.125256,0.12712,0.137249,0.129471,718288,Amazon ARIPL,SAFF GOLD,0.994832,1.0,138865.260689,0.133866,9.640,0.133866,2025-12-31,0.35247,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,2026-01-31,None,Saffola Oils,10.487147,10.156462,9.640,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.145630,0.141038,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0,0.0,1,21.77,19.549,0.30231,0.271468,14.723667,12.557667,0.204461,0.174382,138865.260689,0,1,0.126748,0.384173,NaN,NaN,NaN,NaN,1,0.0,NaN,0.0,0,21.77,19.549,0.30231,0.271468,17.926,14.318833,0.24893,0.198839
1,Amazon ARIPL_718288,2023-02-28,14.459973,9.020000,9.154167,7.835346,8.915246,0.200799,0.125256,0.12712,0.108806,0.123802,718288,Amazon ARIPL,SAFF GOLD,0.861631,1.0,138865.260689,0.109912,7.915,0.109912,2025-12-31,0.35247,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,2026-01-31,None,Saffola Oils,8.433829,8.119195,7.915,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.117117,0.112747,0.0,0.0,0.0,0.0,0.133866,0.000000,0.000000,0,0,0.0,1,21.77,19.549,0.30231,0.271468,14.723667,12.557667,0.204461,0.174382,138865.260689,0,1,0.126748,0.384173,NaN,NaN,NaN,NaN,2,0.0,NaN,0.0,0,21.77,19.549,0.30231,0.271468,17.926,14.318833,0.24893,0.198839
2,Amazon ARIPL_718288,2023-03-31,6.190204,9.020000,9.154167,8.960542,9.751109,0.085960,0.125256,0.12712,0.124431,0.135409,718288,Amazon ARIPL,SAFF GOLD,1.177605,1.0,138865.260689,0.131991,9.505,0.131991,2025-12-31,0.35247,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,2026-01-31,None,Saffola Oils,9.570508,9.278415,9.505,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.132901,0.128845,0.0,0.0,0.0,0.0,0.109912,0.133866,0.000000,0,0,0.0,1,21.77,19.549,0.30231,0.271468,14.723667,12.557667,0.204461,0.174382,138865.260689,0,1,0.126748,0.384173,NaN,NaN,NaN,NaN,3,0.0,NaN,0.0,0,21.77,19.549,0.30231,0.271468,17.926,14.318833,0.24893,0.198839
3,Amazon ARIPL_718288,2023-04-30,8.606029,9.020000,9.154167,7.905760,8.989151,0.119508,0.125256,0.12712,0.109784,0.124828,718288,Amazon ARIPL,SAFF GOLD,0.834181,2.0,138865.260689,0.129006,9.290,0.129006,2025-12-31,0.35247,run,acuuracy_check,/data/aman_singh/acuuracy_check/backtest_files...,0.0,0.0,2026-01-31,None,Saffola Oils,8.478038,8.177136,9.290,9.020000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,9.020000,0.125256,0.0,0.0,0.0,0.117730,0.113552,0.0,0.0,0.0,0.0,0.131991,0.109912,0.133866,0,0,0.0,1,21.77,19.549,0.30231,

In [490]:
# mnth = 7
# all_brand = final_df.groupby(['brand_code', 'run_month_x','month_date'])['vol_in_rum_value'].sum().reset_index()
# def detect_april_anomaly(df, brand_code, threshold=0.25, months_window=3):
#     """
#     Detect if April's vol_in_rum_value is >25% different 
#     from past 3 months & next 3 months, and if pattern repeats in last 2 years.
    
#     Parameters:
#     - df: input dataframe with 'month_date', 'vol_in_rum_value', 'run_month'
#     - brand_code: filter by this brand code
#     - threshold: 25% difference threshold
#     - months_window: number of months before and after April to compare
    
#     """
    
#     # Filter for brand and sort by month_date
#     df_brand = df[df['brand_code'] == brand_code].sort_values('month_date').copy()
    
#     if df_brand.empty:
#         return None
    
#     # Extract year and month
#     df_brand['year'] = df_brand['month_date'].dt.year
#     df_brand['month'] = df_brand['month_date'].dt.month
    
#     # Get unique years (excluding current year if incomplete)
#     years = sorted(df_brand['year'].unique())
#     current_year = years[-1]
#     past_years = [y for y in years if y < current_year][-2:]  # Last 2 years
    
#     anomalies = []
    
#     # Check each past year's April
#     for year in past_years:
#         df_year = df_brand[df_brand['year'] == year].sort_values('month_date')
        
#         # Get April data (month == 4)
#         april_data = df_year[df_year['month'] == mnth]
#         if april_data.empty:
#             continue
        
#         april_value = april_data['vol_in_rum_value'].iloc[0]
#         april_month = mnth
        
#         # Dynamically calculate past and next months
#         past_months = [(april_month - i - 1) % 12 + 1 for i in range(1,months_window+1)]
#         print(past_months)
#         next_months = [(april_month + i - 1) % 12 + 1 for i in range(1, months_window + 1)]
#         print(next_months)
        
#         # Get past and next months values
#         past_3m = df_year[df_year['month'].isin(past_months)]['vol_in_rum_value']
#         next_3m = df_year[df_year['month'].isin(next_months)]['vol_in_rum_value']
        
#         # Combine all comparison months
#         comparison_values = pd.concat([past_3m, next_3m])
        
#         if comparison_values.empty:
#             continue
        
#         # Calculate mean of comparison months
#         #mean_value = comparison_values.mean()
        
#         # Calculate percentage difference
#         pct_diffs = []
#         for comp_value in comparison_values:
#             if comp_value != 0:
#                 pct_diff = (april_value - comp_value) / comp_value
#                 pct_diffs.append(pct_diff)
        
#         # April is anomalous if it's >25% different from ALL comparison months
#         # AND all differences have the same sign (all positive or all negative)
#         if pct_diffs:
#             positive_diffs = [p for p in pct_diffs if p > 0]
#             negative_diffs = [p for p in pct_diffs if p < 0]
#             same_sign = len(positive_diffs) == len(pct_diffs) or len(negative_diffs) == len(pct_diffs)
#             is_anomaly = len([p for p in pct_diffs if abs(p) > threshold]) == len(pct_diffs) and same_sign
#         else:
#             is_anomaly = False

#         anomalies.append({
#             'brand_code': brand_code,
#             'year': year,
#             'april_value': april_value,
#             'num_months_compared': len(comparison_values),
#             'pct_diffs_from_each': pct_diffs,
#             'min_pct_diff': min(pct_diffs) * 100 if pct_diffs else None,
#             'max_pct_diff': max(pct_diffs) * 100 if pct_diffs else None,
#             'is_anomaly': is_anomaly,
#             'direction': 'higher' if april_value > comparison_values.mean() else 'lower'
#         })
    
#     # Check if pattern repeats in both years
#     if len(anomalies) == 2:
#         pattern_repeats = anomalies[0]['is_anomaly'] and anomalies[1]['is_anomaly']
#         return pd.DataFrame(anomalies), pattern_repeats
    
#     return pd.DataFrame(anomalies), False


# # Usage: Apply to each brand code
# brands = all_brand['brand_code'].unique()
# results = []

# for brand in brands:
#     df_result, repeats = detect_april_anomaly(all_brand, brand)
#     if df_result is not None and not df_result.empty:
#         df_result['pattern_repeats'] = repeats
#         results.append(df_result)


# anomaly_summary = pd.concat(results, ignore_index=True)
# print(anomaly_summary[anomaly_summary['pattern_repeats'] == True])

In [491]:
# final_brands = anomaly_summary[anomaly_summary['pattern_repeats'] == True].drop_duplicates(subset=['brand_code'])[['brand_code', 'direction', 'min_pct_diff', 'max_pct_diff']]
# final_brands['month_different'] = 1
# final_df = final_df.merge(final_brands[['brand_code', 'month_different']], on = 'brand_code', how = 'left')
# final_df['month_different'].fillna(0, inplace=True)
# final_df

In [500]:
final_df[(final_df['M month'].notna())].to_csv('/data/aman_singh/acuuracy_check/all_combination_ecom_backtest_pred.csv')

In [288]:
final_df.to_csv('/data/aman_singh/acuuracy_check/all_combination_ecom_trend.csv')

In [304]:
final_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'platform_name', 'brand_code',
       'ratio_last_year', 'quarter', 'qtr_ind_rate_x', 'vol_in_rum_value',
       'vol_in_rum_treated', 'vol_in_rum_value_treated', 'train_till', 'cov',
       'run', 'step', 'file_path', 'pred_best_model', 'pred_value_best_model',
       'run_month', 'M month', 'portfolio', 'pred_prophet_70%ile',
       'pred_prophet_60%ile', 'vol_in_rum', 'P3M', 'P6M', 'LY P3M', 'LY P6M',
       'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'pred_prophet_60%ile_value', 'LY', 'LLY',
       'LY value', 'LLY value', '

In [498]:
final_df[(final_df['run_month'] == '2026-04-30') & (final_df['month_date'] == '2026-05-31')]['pred_value_prophet'].sum()

34.95005620029545

In [294]:
final_df['run_month'].unique()

<DatetimeArray>
['2026-01-31 00:00:00', '2026-02-28 00:00:00', '2026-03-31 00:00:00',
 '2026-04-30 00:00:00']
Length: 4, dtype: datetime64[ns]

### some checks

In [274]:
query = f"""select * from {input_table}
where run_month = '2026-06-30' """

data = pd.read_sql(con=dev_conn, sql=query)
data.columns = data.columns.str.lower()
data

,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,indexbpm,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month
0,2023-01-31,Amazon ARIPL,718288,SAFF GOLD,9.640,13.27071,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
1,2023-02-28,Amazon ARIPL,718288,SAFF GOLD,7.915,10.89602,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
2,2023-03-31,Amazon ARIPL,718288,SAFF GOLD,9.505,13.08486,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
3,2023-04-30,Amazon ARIPL,718288,SAFF GOLD,9.290,12.78889,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
4,2023-05-31,Amazon ARIPL,718288,SAFF GOLD,8.050,11.08187,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127998,2026-11-30,Nykaa,811019,PADV_WIPS,0.000,0.00000,1,0,0,0,0,0,0,0,0,0,0,2026-06-30
127999,2026-12-31,Nykaa,811019,PADV_WIPS,0.000,0.00000,1,0,0,0,0,0,0,0,0,0,0,2026-06-30
128000,2027-01-31,Nykaa,811019,PADV_WIPS,0.000,0.00000,1,0,0,0,0,0,0,0,0,0,0,2026-06-30
128001,2027-02-28,Nykaa,811019,PADV_WIPS,0.000,0.00000,1,0,0,0,0,0,0,0,0,0,0,2026-06-30


In [275]:
data['key'] = (
    data['platform_name'].astype(str) + '_' +
    data['parent_material_code'].astype(str)
)
data = data[data['key'].isin(trend_file_df['key'].unique())]

In [276]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.452544,0.765,0.000000,0.000025,0.000056,0.000022,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,496.828458,0.000022,NaN,NaN,0.45,0.000022,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,0.676890,0.547432,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000034,0.000027,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.289509,0.774,0.000022,0.000025,0.000056,0.000014,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,496.828458,0.000000,NaN,NaN,0.00,0.000000,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,0.545380,0.432579,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000027,0.000021,NaN,NaN,NaN,NaN,0.000022,NaN,NaN
2,Amazon RK_718472,2024-01-31,0.172500,0.51,1.125,1.143780,1.116,0.000009,0.000025,0.000056,0.000057,0.000055,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000054,NaN,NaN,1.08,0.000054,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,1.380028,1.239281,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000069,0.000062,NaN,NaN,NaN,NaN,0.000000,0.000022,NaN
3,Amazon RK_718472,2024-02-29,0.678074,0.51,1.125,1.165967,1.296,0.000034,0.000025,0.000056,0.000058,0.000064,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000080,NaN,NaN,1.62,0.000080,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,1.416077,1.293417,1.62,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,0.000070,0.000064,NaN,NaN,NaN,NaN,0.000054,0.000000,0.000022
4,Amazon RK_718472,2024-03-31,0.844506,0.90,1.125,1.783082,1.818,0.000042,0.000045,0.000056,0.000089,0.000090,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000134,NaN,NaN,2.70,0.000134,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,2.008802,1.887959,2.70,0.90,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,0.000100,0.000094,NaN,NaN,NaN,NaN,0.000080,0.000054,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111634,Big Basket_719192,2026-09-30,0.000000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,719192,Big Basket,VEG_CLEAN,NaN,NaN,NaN,NaN,NaN,100.000000,0.000000,NaN,NaN,0.00,0.000000,2026-05-31,4.605489,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,

In [277]:
import pandas as pd

as_of_date = pd.to_datetime("2026-06-30")  # month-end for Feb 2026
data['month_date'] = pd.to_datetime(data['month_date'])
filtered = data[
    
    (data['month_date'] < as_of_date) &
    (data['month_date'] >= as_of_date - pd.DateOffset(months=3))
]


In [278]:
# assert p3m equals
x = filtered.groupby(['month_date'])['vol_in_rum'].sum().reset_index()['vol_in_rum'].mean()
y = trend_file_df[trend_file_df['month_date'] == '2026-06-30']['P3M'].sum()
assert(int(x)==int(y))

In [279]:
(x,y)

(267025.7059451062, 267025.7059451031)

In [280]:
ly_end = as_of_date - pd.DateOffset(years=1)
ly_start = ly_end - pd.DateOffset(months=3)

filtered = data[
    
    (data['month_date'] < ly_end) &
    (data['month_date'] >= ly_start )
]


In [282]:
# p3m ly check may not equal but should be close
x = filtered.groupby(['month_date'])['vol_in_rum'].sum().reset_index()['vol_in_rum'].mean()
y = trend_file_df[trend_file_df['month_date'] == '2026-06-30']['LY P3M'].sum()
(x,y)

(307007.49542052665, 305203.2554095226)

In [283]:
# p3m consistency check
as_of_date = pd.to_datetime('2026-06-30')

next_3_months = pd.date_range(
    start=as_of_date + pd.offsets.MonthEnd(1),
    periods=3,
    freq='M'
)
for dt in next_3_months:
    p3m_sum = trend_file_df.loc[
        trend_file_df['month_date'] == dt, 'P3M'
    ].sum()
    
    print(f"P3M sum for {dt.date()}: {p3m_sum}")


P3M sum for 2026-07-31: 267025.7059451031
P3M sum for 2026-08-31: 267025.7059451031
P3M sum for 2026-09-30: 267025.7059451031


### The end

In [106]:
import numpy as np
import pandas as pd

df = final_df.copy()

# --------------------------------------------------
# 1. SAFE FACTOR FUNCTIONS (NO ERRORS)
# --------------------------------------------------

def safe_div(a, b):
    """Safe division: if error or b<=0 → return 1."""
    try:
        if b is None or b == 0:
            return 1
        return a / b
    except:
        return 1

# recency factor = p3m/p6m (cap at 2)
df["recency_factor"] = df.apply(
    lambda r: min(2, safe_div(r["P3M_value"], r["P6M_value"])),
    axis=1
)

def safe_shrink(r,column_name):
    try:
        ratio = r[column_name] / r["P3M_value"]
        return 1 / np.sqrt(ratio)
    except:
        return 1

df["shrink_ratio_prophet"] = df.apply(lambda r: safe_shrink(r, "pred_value_prophet"), axis=1)
df["shrink_ratio_rf"] = df.apply(lambda r: safe_shrink(r, "pred_value_rf"), axis=1)

# seasonality factor = p3m / p3mLY (cap at 2)
df["seasonality_factor"] = df.apply(
    lambda r: min(2, safe_div(r["P3M_value"], r["LY P3M_value"])),
    axis=1
)


# shrink_ratio = 1 / sqrt(max(forecast/p3m,1)) → safe



# --------------------------------------------------
# 2. HEURISTICS
# --------------------------------------------------

# Recency heuristic: max(recency_factor * P3M, forecast)
df["recency_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M_value"], r["pred_value_prophet"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'], r["pred_value_prophet"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'] * r["recency_factor"],
                  r["pred_value_prophet"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_prophet_value"] = df["pred_value_prophet"] * df["shrink_ratio_prophet"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_prophet_value"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_prophet_value"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_prophet_value"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_prophet_value"]


df["final_heuristic_prophet_value"] = df.apply(apply_final_logic, axis=1)


In [107]:
df["recency_heuristic_rf_value"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M_value"], r["pred_value_rf"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_rf_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'], r["pred_value_rf"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_rf_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'] * r["recency_factor"],
                  r["pred_value_rf"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_rf_value"] = df["pred_value_rf"] * df["shrink_ratio_rf"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_rf_value"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_rf_value"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_rf_value"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_rf_value"]


df["final_heuristic_rf_value"] = df.apply(apply_final_logic, axis=1)


In [108]:
df["error_prophet"] = df["pred_value_prophet"] - df["vol_in_rum_value"]
df["abs_error_prophet"] = df["error_prophet"].abs()

df["error_rf"] = df["pred_value_rf"] - df["vol_in_rum_value"]
df["abs_error_rf"] = df["error_rf"].abs()

df["error_final_heuristic_prophet"] = df["final_heuristic_prophet_value"] - df["vol_in_rum_value"]
df["abs_error_final_heuristic_prophet"] = df["error_final_heuristic_prophet"].abs()

df["error_final_heuristic_rf"] = df["final_heuristic_rf_value"] - df["vol_in_rum_value"]
df["abs_error_final_heuristic_rf"] = df["error_final_heuristic_rf"].abs()


In [109]:
df["recency_heuristic_rf"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M"], r["pred_rf"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_rf"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'], r["pred_rf"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_rf"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'] * r["recency_factor"],
                  r["pred_rf"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_rf"] = df["pred_rf"] * df["shrink_ratio_rf"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_rf"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_rf"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_rf"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_rf"]


df["final_heuristic_rf"] = df.apply(apply_final_logic, axis=1)


In [110]:
df["recency_heuristic_prophet"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M"], r["pred_prophet"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_prophet"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'], r["pred_prophet"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_prophet"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'] * r["recency_factor"],
                  r["pred_prophet"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_prophet"] = df["pred_prophet"] * df["shrink_ratio_prophet"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_prophet"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_prophet"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_prophet"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_prophet"]


df["final_heuristic_prophet"] = df.apply(apply_final_logic, axis=1)


In [112]:
df.to_csv('Heuristics_all_combination_ecom_cp.csv')

In [114]:
df[(df['M month'].notna())].to_csv('Heuristics_all_combination_ecom_cp2.csv')

In [2]:
import pandas as pd
final_df = pd.read_csv('Heuristics_all_combination_ecom_cp.csv')

/tmp/ipykernel_2052077/3783379672.py:2: DtypeWarning: Columns (20,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv('Heuristics_all_combination_ecom_cp.csv')


In [6]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["vol_in_rum_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,platform_name,parent_material_code,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,Amazon ARIPL,718288,2025-12-31,0.134916,0.345766,0.219256,0.042170
1,Amazon ARIPL,718321,2025-12-31,0.000000,0.000000,0.000000,0.000000
2,Amazon ARIPL,718322,2025-12-31,0.060035,0.108314,0.079347,0.009656
3,Amazon ARIPL,718323,2025-12-31,0.000000,0.000000,0.000000,0.000000
4,Amazon ARIPL,718328,2025-12-31,0.016588,0.032115,0.022799,0.003105


In [7]:
threshold_df.to_csv('ecom_threshold.csv')

In [162]:
final_df[(final_df['M month'].notna())].to_csv('Heuristics_all_combination_qcom_cp_chk.csv')

In [129]:
final_df[final_df['month_date'] == '2026-02-28']['LY P3M_value'].sum()

123.92211585241307

In [49]:
df['TREND'].unique()

array([ 1,  0, -1])

In [96]:
prophet_output = collate_file('prophet_data_train_till')

downloaded_results\new_pipeline\202508-09_FK_Others_Offtakes\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\new_pipeline\202508-09_FK_Others_Offtakes\train_till_31_Jul_2025\prophet_results\prophet_data_train_till_31_Jul_2025.csv
downloaded_results\new_pipeline\202509_AZ_BB_Offtakes\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\new_pipeline\202509_AZ_BB_Offtakes\train_till_31_Jul_2025\prophet_results\prophet_data_train_till_31_Jul_2025.csv


In [100]:
len_before_merge = len(offtake_df)
offtake_df = offtake_df.merge(
    read_qtr_ind_rate_table()[['brand_code', 'qtr_ind_rate']] ,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(offtake_df)


Credentials retrieved successfully for prod db.


In [ ]:
offtake_df

In [104]:
offtake_df['OT_Value_in_Cr'] = offtake_df['vol_in_rum'] * offtake_df['qtr_ind_rate'] / (10 ** 7)

In [105]:
offtake_df.to_csv('Offtake_realigned_base.csv', index=False)

In [108]:
trend_file_df[trend_file_df['run_month'].isin(['2025-10-31'])].to_csv('Trend_File_Oct_Live_Run_OT.csv', index=False)

In [107]:
trend_file_df[trend_file_df['run_month'].isin(['2025-07-31', '2025-08-31'])].to_csv('Trend_File_SepAug_OT.csv', index=False)

In [97]:
prophet_output.to_csv('ECOM_Prophet_Trend_OT_Chain_PSKU_AugSep.csv', index=False)

In [81]:
forecast_df = trend_file_df[trend_file_df['month_date'] >= trend_file_df['run_month']]

forecast_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,LY P6M,P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,LY,LLY,LY value,LLY value
26,Amazon_718589,2025-03-31,1312.900000,1293.400000,1824.634404,945.661643,0.059090,0.058213,0.082122,0.042562,...,841.200000,0.059090,0.058213,0.027878,0.037860,0.082122,717.3,1428.0,0.032284,0.064271
27,Amazon_718589,2025-04-30,1312.900000,1293.400000,1894.995393,1306.938750,0.059090,0.058213,0.085289,0.058822,...,792.800000,0.059090,0.058213,0.036843,0.035682,0.085289,1053.0,1040.4,0.047393,0.046826
28,Amazon_718589,2025-05-31,1312.900000,1293.400000,2055.487818,1194.591429,0.059090,0.058213,0.092512,0.053765,...,682.250000,0.059090,0.058213,0.039944,0.030706,0.092512,1241.1,1236.3,0.055859,0.055643
29,Amazon_718589,2025-06-30,1312.900000,1293.400000,1713.411024,989.591786,0.059090,0.058213,0.077116,0.044539,...,811.600000,0.059090,0.058213,0.045178,0.036528,0.077116,797.4,1176.0,0.035889,0.052929
56,Amazon_722188,2025-03-31,0.666667,6.400000,0.000000,471.682496,0.000030,0.000288,0.000000,0.021229,...,782.533333,0.000030,0.000288,0.043045,0.035220,0.000000,806.4,236.8,0.036294,0.010658
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120425,Flipkart National_809422,2025-10-31,211.833333,159.550000,262.584032,129.597491,0.030576,0.023029,0.037901,0.018706,...,207.383333,0.030576,0.023029,0.009040,0.029934,0.046132,0.8,NaN,0.000115,NaN
120442,Flipkart National_809423,2025-07-31,75.000000,53.533333,0.000000,169.044957,0.010826,0.007727,0.000000,0.024400,...,NaN,0.010826,0.007727,0.057582,NaN,0.008194,243.4,NaN,0.035132,NaN
120443,Flipkart National_809423,2025-08-31,75.000000,53.533333,0.000000,117.175693,0.010826,0.007727,0.000000,0.016913,...,NaN,0.010826,0.007727,0.054599,NaN,0.000000,99.6,NaN,0.014376,NaN
120444,Flipkart National_809423,2025-09-30,75.000000,53.533333,0.000000,111.920263,0.010826,0.007727,0.000000,0.016155,...,286.050000,0.010826,0.007727,0.048200,0.041288,0.000000,75.4,NaN,0.010883,NaN


In [98]:
trend_file_df[trend_file_df['run_month'] == '2025-08-31'].to_csv('Trend_file_Sep.csv', index=False)

In [82]:
forecast_df['is_na'] = forecast_df['vol_in_rum'].isna()
forecast_df.groupby(['run_month', 'month_date'])['is_na'].sum()

run_month   month_date
2025-03-31  2025-03-31    0
            2025-04-30    0
            2025-05-31    0
            2025-06-30    0
2025-04-30  2025-04-30    0
            2025-05-31    0
            2025-06-30    0
            2025-07-31    0
2025-05-31  2025-05-31    0
            2025-06-30    0
            2025-07-31    0
            2025-08-31    0
2025-06-30  2025-06-30    0
            2025-07-31    0
            2025-08-31    0
            2025-09-30    0
2025-07-31  2025-07-31    0
            2025-08-31    0
            2025-09-30    0
            2025-10-31    0
Name: is_na, dtype: int64

In [83]:
forecast_df.drop('is_na', axis=1, inplace=True)

In [84]:
pred_value_cols = [col for col in trend_file_df.columns if 'value' in col and 'pred' in col]
pred_value_cols

['pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'pred_value_best_model',
 'pred_prophet_70%ile_value']

In [85]:
for col in pred_value_cols:
    trend_file_df[f'error_{col}'] = trend_file_df[col] - trend_file_df['vol_in_rum_value']
    trend_file_df[f'abs_error_{col}'] = np.abs(trend_file_df[col] - trend_file_df['vol_in_rum_value'])    

In [86]:
trend_file_df.to_csv('Trend_file_OT_FK_AZ_BB.csv', index=False)

In [85]:
feature_importance_df = collate_file('feature_importance_train_till')

downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_28_Feb_2025\ml_results\feature_importance_train_till_28_Feb_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_30_Apr_2025\ml_results\feature_importance_train_till_30_Apr_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_30_Jun_2025\ml_results\feature_importance_train_till_30_Jun_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_31_Mar_2025\ml_results\feature_importance_train_till_31_Mar_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_31_May_2025\ml_results\feature_importance_train_till_31_May_2025.csv


In [87]:
feature_importance_df.to_excel('202505-AZ_BB_OT_Feature_Imp.xlsx', index=False)

In [11]:
collate_file('prophet_data_train_till_').to_excel('202504-08_Prophet_File.xlsx', index=False)

downloaded_results\ECOM_ChainPSKU_OT_run\train_till_28_Feb_2025\prophet_results\prophet_data_train_till_28_Feb_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_30_Apr_2025\prophet_results\prophet_data_train_till_30_Apr_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_31_Mar_2025\prophet_results\prophet_data_train_till_31_Mar_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_31_May_2025\prophet_results\prophet_data_train_till_31_May_2025.csv


### Secondary

In [87]:
sec_query = """SELECT 
    chain,
    parent_material_code, 
    month_date, 
    SUM(sec_actuals_vol_rum_month) AS sec_vol_actuals_rum_month,
    SUM(sec_apo_plan_vol_rum_month) AS sec_apo_plan_vol_rum_month
FROM (
    SELECT 
        month_date, 
        distributor_code, 
        material_code, 
        sec_actuals_vol_rum_month, 
        sec_apo_plan_vol_rum_month
    FROM 
        dwh_bpm_dist_brand_mth_sbp 
    WHERE 
        month_date BETWEEN '2022-04-01' AND '2025-12-31'
) A
JOIN (
    SELECT DISTINCT
        customer, 
        chain
    FROM 
        mst_chain_master 
    WHERE 
        chain_type = 'E Com B2C'
) CC
    ON A.distributor_Code = CC.customer
JOIN (
    SELECT 
        material_code, 
        parent_material_code 
    FROM 
        mst_material 
    WHERE 
        company_code = 'MIL' 
        AND latest_record_ind = 1
) M 
    ON A.material_code = M.material_code
GROUP BY 
    chain,
    parent_material_code, 
    month_date
ORDER BY 
    chain,
    parent_material_code, 
    month_date;
"""


results = pd.read_sql(con=prod_conn, sql=sec_query)
sales_data = pd.DataFrame(results)
sales_data.columns = sales_data.columns.str.lower()
sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})

In [88]:
sales_data['month_date'] = pd.to_datetime(sales_data['month_date'])

In [89]:
sales_data['chain'] = sales_data['chain'].replace({
    'Grofers': 'Blinkit',
    'Amazon B2C': 'Amazon ARIPL',
    'Flipkart-Grocery': 'Flipkart Grocery',
    'FlipkartGrocery': 'Flipkart Grocery',
    'Flipkart-National': 'Flipkart National',
    'RK WORLDINFOCOM': 'Amazon RK',
    'ZEPTO': 'Zepto',
    'Kiranakart Technologies': 'Zepto',
    'Big basket B2B': 'Big Basket',
    'Big basket B2C': 'Big Basket',
    'Myntra': 'MYNTRA'
})

In [90]:
chains = ['Big Basket', 'Blinkit', 'Amazon ARIPL', 'Flipkart Grocery',
       'Flipkart National', 'Swiggy', 'Zepto', 'Amazon RK', 'City Mall',
       '1MG', 'Dealshare', 'Meesho', 'Nykaa', 'First Cry', 'MYNTRA',
       'Purplle']

In [91]:
for chain in chains:
    if chain not in sales_data['chain'].unique():
        print(chain)

In [92]:
dev_conn = get_dbconnection('DEV')
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment_2""",
    dev_conn
)

realignment_df.columns = realignment_df.columns.str.lower()

def realign_pskus(data, channel='ECOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "All")
    ]
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']
    
    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data['parent_material_code'] == old_psku, "parent_material_code"
        ] = new_psku

    return data


Credentials retrieved successfully for dev db.


In [93]:
realigned_df = realign_pskus(sales_data.copy())

In [94]:
realigned_df = realigned_df.groupby(
    ['chain', 'parent_material_code', 'month_date'], as_index=False
).sum()

In [95]:
old_pskus = realignment_df[
    (realignment_df["channel"] == "ECOM")
    | (realignment_df["channel"] == "ECOM" + " B2C")
    | (realignment_df["channel"] == "All")
]['psku old'].unique()

for psku in old_pskus:
    assert psku not in realigned_df['parent_material_code'].unique()

In [96]:
realigned_df['key'] = realigned_df['chain'] + '_' + realigned_df['parent_material_code'].astype(str) 
realigned_df['month_date'] = pd.to_datetime(realigned_df['month_date'])

realigned_df.duplicated(subset=['key', 'month_date']).sum()

0

In [97]:
realigned_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

0

In [99]:
realigned_df


,chain,parent_material_code,month_date,sec_vol_actuals_rum_month,sec_apo_plan_vol_rum_month,key
0,1MG,715095,2024-01-31,0.0,0.000,1MG_715095
1,1MG,715096,2023-05-31,0.0,0.000,1MG_715096
2,1MG,715096,2023-06-30,0.0,0.189,1MG_715096
3,1MG,715096,2023-07-31,0.0,0.016,1MG_715096
4,1MG,715096,2023-08-31,0.0,0.000,1MG_715096
...,...,...,...,...,...,...
1081559,imli,807033,2025-04-30,0.0,0.000,imli_807033
1081560,imli,807033,2025-05-31,0.0,0.000,imli_807033
1081561,imli,807033,2025-06-30,0.0,0.000,imli_807033
1081562,imli,807033,2025-07-31,0.0,0.000,imli_807033


In [105]:
trend_file_df['platform_name'].unique()

array(['Amazon', 'Big Basket', 'Flipkart Grocery', 'Flipkart National'],
      dtype=object)

In [106]:
trend_file_df['platform_updated'] = np.where(
    trend_file_df['platform_name'] == 'Amazon', 
    np.where(
        trend_file_df['portfolio'].isin(['Foods', 'Saffola Oils']), 
        'Amazon ARIPL', 
        'Amazon RK'
    ),
    trend_file_df['platform_name']
)

In [108]:
trend_file_df['platform_updated'].unique()

array(['Amazon RK', 'Big Basket', 'Flipkart Grocery', 'Flipkart National',
       'Amazon ARIPL'], dtype=object)

In [109]:
trend_file_df.groupby(
    'platform_name'
)['platform_updated'].unique()

platform_name
Amazon               [Amazon RK, Amazon ARIPL]
Big Basket                        [Big Basket]
Flipkart Grocery            [Flipkart Grocery]
Flipkart National          [Flipkart National]
Name: platform_updated, dtype: object

In [102]:
trend_file_df['portfolio']

0             Hair Oils
1             Hair Oils
2             Hair Oils
3             Hair Oils
4             Hair Oils
              ...      
120441    Male Grooming
120442    Male Grooming
120443    Male Grooming
120444    Male Grooming
120445    Male Grooming
Name: portfolio, Length: 120446, dtype: object

In [113]:
trend_file_df['key'] = trend_file_df[
    ['platform_updated', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [116]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    realigned_df[['key', 'month_date', 'sec_vol_actuals_rum_month']],
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(trend_file_df)
del len_before_merge